In [4]:
"""
Unified LoRA Fine-Tuning for Syllogistic Reasoning
=====================================================
4 strategies selectable via STRATEGY flag:

  "simple"        = Option 1: validity classification only
  "contrastive"   = Option 2: validity + plausibility-invariance pairs
  "orthogonal"    = Option 3: multi-task with orthogonal separation
  "adversarial"   = Option 4: multi-task with gradient reversal

Fixes applied:
  [1] Single AutoModelForCausalLM for both generation loss and representations.
      We use logits for causal LM loss and hidden_states (pre-lm_head) for
      representation learning. No separate AutoModel needed.
  [2] Loss normalization: each loss component is scale-normalized before
      weighting so no single term dominates.
  [3] Improved contrastive pairing: cross-validity pairs added as negative
      examples (same plausibility, different validity) for harder contrast.
  [4] Weighted attention pooling: last_token still default for decoder,
      but added attention_weighted option using attention scores as weights.
"""

import os
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
import importlib.util

# ============================================================
# CONFIG
# ============================================================
STRATEGY = "simple"  # "simple" | "contrastive" | "orthogonal" | "adversarial"

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUTPUT_DIR = f"lora_{STRATEGY}_output"
HF_TOKEN = os.getenv("HF_TOKEN", "")

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

EPOCHS = 5
BATCH_SIZE = 4
GRAD_ACCUM = 4
LR = 2e-4
HEAD_LR = 5e-4
WARMUP_RATIO = 0.1
MAX_LEN = 512
SEED = 42

# Contrastive
LAMBDA_CONTRASTIVE = 0.5
CONTRASTIVE_LAYER_FRAC = 0.25
USE_NEGATIVE_PAIRS = True  # [FIX 3] add cross-validity negative pairs

# Orthogonal/Adversarial
LAMBDA_PLAUS = 0.3
LAMBDA_ORTH = 0.3          # [FIX 2] reduced from 1.0, gets normalized anyway
LAMBDA_DECORR = 0.1
LAMBDA_VAL_HEAD = 0.3
GRL_LAMBDA = 1.0
HEAD_PROJ_DIM = 256
EXTRACT_LAYER = -1

# [FIX 4] "last_token" | "mean_pool" | "attention_weighted"
POOL_MODE = "last_token"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def load_json(p):
    with open(p, "r", encoding="utf-8") as f: return json.load(f)

def save_json(o, p):
    os.makedirs(os.path.dirname(p) or ".", exist_ok=True)
    with open(p, "w", encoding="utf-8") as f: json.dump(o, f, indent=2, ensure_ascii=False)


# ============================================================
# DATASETS
# ============================================================

class SimpleDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def _prompt(self, syl):
        msgs = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syl}\n\nAnswer yes or no."}
        ]
        return self.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def __getitem__(self, idx):
        ex = self.data[idx]
        prompt = self._prompt(ex["syllogism"])
        target = " yes" if ex["validity"] else " no"
        full = prompt + target
        enc = self.tokenizer(full, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
        penc = self.tokenizer(prompt, truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": enc.input_ids.squeeze(0),
            "attention_mask": enc.attention_mask.squeeze(0),
            "prompt_len": penc.input_ids.shape[1],
            "validity": 1 if ex["validity"] else 0,
            "plausibility": 1 if ex["plausibility"] else 0,
        }


# [FIX 3] Improved pairing with negative examples
class PairDataset(Dataset):
    def __init__(self, data, tokenizer, max_len, use_negatives=True):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.pairs = []

        vp = [x for x in data if x["validity"] and x["plausibility"]]
        vi = [x for x in data if x["validity"] and not x["plausibility"]]
        ip = [x for x in data if not x["validity"] and x["plausibility"]]
        ii = [x for x in data if not x["validity"] and not x["plausibility"]]

        rng = random.Random(SEED)

        # Positive pairs: same validity, different plausibility → push together
        def pos_pair(a, b, val):
            rng.shuffle(a); rng.shuffle(b)
            n = min(len(a), len(b))
            for i in range(n):
                self.pairs.append({
                    "syl_a": a[i]["syllogism"], "syl_b": b[i]["syllogism"],
                    "validity": val, "pair_type": "positive"
                })

        pos_pair(list(vp), list(vi), True)
        pos_pair(list(ip), list(ii), False)

        # Negative pairs: same plausibility, different validity → push apart
        if use_negatives:
            def neg_pair(a, b):
                rng.shuffle(a); rng.shuffle(b)
                n = min(len(a), len(b))
                for i in range(n):
                    self.pairs.append({
                        "syl_a": a[i]["syllogism"], "syl_b": b[i]["syllogism"],
                        "validity_a": a[i]["validity"], "validity_b": b[i]["validity"],
                        "pair_type": "negative"
                    })
            neg_pair(list(vp), list(ip))  # plausible: valid vs invalid
            neg_pair(list(vi), list(ii))  # implausible: valid vs invalid

        rng.shuffle(self.pairs)
        n_pos = sum(1 for p in self.pairs if p["pair_type"] == "positive")
        n_neg = sum(1 for p in self.pairs if p["pair_type"] == "negative")
        print(f"Pairs: {n_pos} positive (push together), {n_neg} negative (push apart)")

    def __len__(self): return len(self.pairs)

    def _prompt(self, syl):
        msgs = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syl}\n\nAnswer yes or no."}
        ]
        return self.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def __getitem__(self, idx):
        p = self.pairs[idx]
        out = {"pair_type": 0 if p["pair_type"] == "positive" else 1}

        if p["pair_type"] == "positive":
            target_a = " yes" if p["validity"] else " no"
            target_b = target_a
        else:
            target_a = " yes" if p["validity_a"] else " no"
            target_b = " yes" if p["validity_b"] else " no"

        for suffix, syl, target in [("_a", p["syl_a"], target_a), ("_b", p["syl_b"], target_b)]:
            prompt = self._prompt(syl)
            full = prompt + target
            enc = self.tokenizer(full, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
            penc = self.tokenizer(prompt, truncation=True, max_length=self.max_len, return_tensors="pt")
            out[f"input_ids{suffix}"] = enc.input_ids.squeeze(0)
            out[f"attention_mask{suffix}"] = enc.attention_mask.squeeze(0)
            out[f"prompt_len{suffix}"] = penc.input_ids.shape[1]

        return out


# ============================================================
# HEADS & GRADIENT REVERSAL
# ============================================================

class GRLFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.clone()
    @staticmethod
    def backward(ctx, grad):
        return -ctx.lam * grad, None

class GRL(nn.Module):
    def __init__(self, lam=1.0): super().__init__(); self.lam = lam
    def forward(self, x): return GRLFunction.apply(x, self.lam)

class DisentangleHeads(nn.Module):
    def __init__(self, hidden_dim, proj_dim, use_grl=False, grl_lam=1.0):
        super().__init__()
        self.val_proj = nn.Sequential(nn.Linear(hidden_dim, proj_dim), nn.ReLU(), nn.Dropout(0.1))
        self.val_cls = nn.Linear(proj_dim, 2)
        self.plaus_proj = nn.Sequential(nn.Linear(hidden_dim, proj_dim), nn.ReLU(), nn.Dropout(0.1))
        self.plaus_cls = nn.Linear(proj_dim, 2)
        self.grl = GRL(grl_lam) if use_grl else nn.Identity()

    def forward(self, h):
        hv = self.val_proj(h)
        hp = self.plaus_proj(self.grl(h))
        return self.val_cls(hv), self.plaus_cls(hp), hv, hp


# ============================================================
# LOSS HELPERS
# ============================================================

def causal_lm_loss(logits, input_ids, attention_mask, prompt_len):
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()
    mask = torch.zeros_like(shift_labels, dtype=torch.float32)
    for i in range(input_ids.shape[0]):
        pl = prompt_len[i].item() if isinstance(prompt_len, torch.Tensor) else prompt_len[i]
        end = attention_mask[i].sum().item() - 1
        mask[i, max(pl - 1, 0):end] = 1.0
    loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), reduction="none").view(shift_labels.shape)
    return (loss * mask).sum() / (mask.sum() + 1e-8)


# [FIX 3] Updated to handle positive (push together) and negative (push apart) pairs
def contrastive_repr_loss(hidden_a, hidden_b, mask_a, mask_b, layer_frac, pair_type):
    n_layers = len(hidden_a)
    start = int(n_layers * (1.0 - layer_frac))
    total = 0.0
    count = 0
    for li in range(start, n_layers):
        ha, hb = hidden_a[li], hidden_b[li]
        pos_a = mask_a.sum(dim=1) - 1
        pos_b = mask_b.sum(dim=1) - 1
        ra = ha[torch.arange(ha.size(0)), pos_a].float()
        rb = hb[torch.arange(hb.size(0)), pos_b].float()
        cos_sim = F.cosine_similarity(ra, rb, dim=-1)

        # Per-example: positive pairs → minimize distance, negative → maximize distance
        # pair_type: 0=positive, 1=negative
        pt = pair_type.float()
        # positive: loss = 1 - cos_sim  (push together)
        # negative: loss = max(0, cos_sim - margin)  (push apart)
        margin = 0.2
        pos_loss = (1.0 - cos_sim) * (1.0 - pt)
        neg_loss = torch.clamp(cos_sim - margin, min=0.0) * pt
        total += (pos_loss + neg_loss).mean()
        count += 1
    return total / max(count, 1)


def orth_loss(hv, hp):
    hv_n = F.normalize(hv, dim=-1)
    hp_n = F.normalize(hp, dim=-1)
    cc = (hv_n.T @ hp_n) / hv.shape[0]
    return torch.sum(cc ** 2)


def decorr_loss(hv, hp):
    def off_diag(h):
        hn = F.normalize(h, dim=0)
        c = hn.T @ hn / h.shape[0]
        m = ~torch.eye(c.shape[0], dtype=torch.bool, device=c.device)
        return torch.sum(c[m] ** 2)
    return off_diag(hv) + off_diag(hp)


# [FIX 4] Pooling with attention_weighted option
def pool_repr(hidden_states, attention_mask, layer_idx=-1, mode="last_token", attentions=None):
    h = hidden_states[layer_idx]  # (batch, seq, hidden)

    if mode == "last_token":
        pos = attention_mask.sum(dim=1) - 1
        return h[torch.arange(h.size(0), device=h.device), pos].float()

    elif mode == "mean_pool":
        mask = attention_mask.unsqueeze(-1).float()
        return ((h * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-8)).float()

    elif mode == "attention_weighted":
        # Use attention weights from last layer as importance scores
        # attentions[-1] shape: (batch, heads, seq, seq)
        if attentions is not None and len(attentions) > 0:
            attn = attentions[-1]  # last layer attention
            # Average across heads, take attention TO the last token
            last_pos = attention_mask.sum(dim=1) - 1  # (batch,)
            # Gather attention weights for last token attending to all positions
            attn_weights = torch.zeros(h.size(0), h.size(1), device=h.device)
            for i in range(h.size(0)):
                lp = last_pos[i].item()
                # Mean across heads: how much the last token attends to each position
                attn_weights[i, :lp+1] = attn[i, :, lp, :lp+1].mean(dim=0)
            # Normalize
            attn_weights = attn_weights * attention_mask.float()
            attn_weights = attn_weights / (attn_weights.sum(dim=1, keepdim=True) + 1e-8)
            # Weighted sum
            return (h * attn_weights.unsqueeze(-1)).sum(dim=1).float()
        else:
            # Fallback to last_token if no attention weights
            pos = attention_mask.sum(dim=1) - 1
            return h[torch.arange(h.size(0), device=h.device), pos].float()

    raise ValueError(f"Unknown pool mode: {mode}")


# [FIX 2] Loss normalizer: tracks running stats to keep losses on similar scales
class LossNormalizer:
    """
    Tracks EMA of each loss component's magnitude.
    Divides each loss by its EMA so all components are ~1.0 before weighting.
    First warmup_steps use simple averaging instead of EMA for stability.
    """
    def __init__(self, keys, momentum=0.9, warmup_steps=50):
        self.momentum = momentum
        self.warmup_steps = warmup_steps
        self.ema = {k: 1.0 for k in keys}
        self.initialized = {k: False for k in keys}
        self.step_count = {k: 0 for k in keys}
        self.warmup_sum = {k: 0.0 for k in keys}

    def normalize(self, key, loss_val):
        v = loss_val.detach().item()
        if v < 1e-8:
            return loss_val
        self.step_count[key] += 1
        if self.step_count[key] <= self.warmup_steps:
            self.warmup_sum[key] += v
            self.ema[key] = self.warmup_sum[key] / self.step_count[key]
            self.initialized[key] = True
        else:
            self.ema[key] = self.momentum * self.ema[key] + (1 - self.momentum) * v
            self.initialized[key] = True
        return loss_val / (self.ema[key] + 1e-8)


# ============================================================
# TRAINING
# ============================================================

def train():
    set_seed(SEED)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    token = HF_TOKEN or None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Strategy: {STRATEGY}")
    print(f"Model: {MODEL_NAME}")
    print(f"Pool mode: {POOL_MODE}")

    # [FIX 1] Single AutoModelForCausalLM for both generation loss and representations.
    # We use logits for causal LM loss and hidden_states (pre-lm_head) for
    # representation learning via the disentanglement heads.
    # No separate AutoModel needed — hidden_states are encoder representations.
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, token=token,
        torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    lora_cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
                          lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGET_MODULES, bias="none")
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    train_data = load_json(TRAIN_JSON)
    use_heads = STRATEGY in ("orthogonal", "adversarial")
    use_pairs = STRATEGY == "contrastive"
    need_attn = (POOL_MODE == "attention_weighted")

    if use_pairs:
        dataset = PairDataset(train_data, tokenizer, MAX_LEN, use_negatives=USE_NEGATIVE_PAIRS)
    else:
        dataset = SimpleDataset(train_data, tokenizer, MAX_LEN)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

    heads = None
    param_groups = [{"params": model.parameters(), "lr": LR, "weight_decay": 0.01}]
    if use_heads:
        heads = DisentangleHeads(
            model.config.hidden_size, HEAD_PROJ_DIM,
            use_grl=(STRATEGY == "adversarial"), grl_lam=GRL_LAMBDA
        ).to(DEVICE)
        param_groups.append({"params": heads.parameters(), "lr": HEAD_LR, "weight_decay": 0.01})
        print(f"Head params: {sum(p.numel() for p in heads.parameters()):,}")

    optimizer = torch.optim.AdamW(param_groups)
    total_steps = (len(dataloader) * EPOCHS) // GRAD_ACCUM
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)

    # [FIX 2] Initialize loss normalizer
    loss_keys = ["cls", "con", "vh", "ph", "ort", "dec"]
    loss_norm = LossNormalizer(loss_keys)

    print(f"Dataset: {len(dataset)} examples, {len(dataloader)} batches/epoch")
    print(f"Total steps: {total_steps}\n")

    model.train()
    if heads: heads.train()
    history = []

    for epoch in range(EPOCHS):
        m = {"cls": 0, "con": 0, "val_h": 0, "plaus_h": 0, "ort": 0, "dec": 0, "total": 0, "n": 0}

        for bi, batch in enumerate(dataloader):
            # ---- SIMPLE / ORTHOGONAL / ADVERSARIAL ----
            if not use_pairs:
                ids = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                plen = batch["prompt_len"]
                val_lbl = batch["validity"].to(DEVICE)
                plaus_lbl = batch["plausibility"].to(DEVICE)

                out = model(ids, attention_mask=mask,
                            output_hidden_states=use_heads,
                            output_attentions=need_attn,
                            use_cache=False)

                cls = causal_lm_loss(out.logits, ids, mask, plen)
                # [FIX 2] Normalize before weighting
                loss = loss_norm.normalize("cls", cls) * 1.0
                m["cls"] += cls.item() * ids.shape[0]

                if use_heads:
                    attns = out.attentions if need_attn else None
                    rep = pool_repr(out.hidden_states, mask, EXTRACT_LAYER, POOL_MODE, attns)
                    vl, pl, hv, hp = heads(rep)
                    vh_loss = F.cross_entropy(vl, val_lbl)
                    ph_loss = F.cross_entropy(pl, plaus_lbl)
                    ol = orth_loss(hv, hp)
                    dl = decorr_loss(hv, hp)

                    # [FIX 2] Each component normalized then weighted
                    loss = loss + LAMBDA_VAL_HEAD * loss_norm.normalize("vh", vh_loss)
                    loss = loss + LAMBDA_PLAUS * loss_norm.normalize("ph", ph_loss)
                    loss = loss + LAMBDA_ORTH * loss_norm.normalize("ort", ol)
                    loss = loss + LAMBDA_DECORR * loss_norm.normalize("dec", dl)

                    m["val_h"] += vh_loss.item() * ids.shape[0]
                    m["plaus_h"] += ph_loss.item() * ids.shape[0]
                    m["ort"] += ol.item() * ids.shape[0]
                    m["dec"] += dl.item() * ids.shape[0]

                m["total"] += loss.item() * ids.shape[0]
                m["n"] += ids.shape[0]

            # ---- CONTRASTIVE ----
            else:
                ids_a = batch["input_ids_a"].to(DEVICE)
                mask_a = batch["attention_mask_a"].to(DEVICE)
                plen_a = batch["prompt_len_a"]
                ids_b = batch["input_ids_b"].to(DEVICE)
                mask_b = batch["attention_mask_b"].to(DEVICE)
                plen_b = batch["prompt_len_b"]
                pair_type = batch["pair_type"].to(DEVICE)

                out_a = model(ids_a, attention_mask=mask_a, output_hidden_states=True, use_cache=False)
                out_b = model(ids_b, attention_mask=mask_b, output_hidden_states=True, use_cache=False)

                cls_a = causal_lm_loss(out_a.logits, ids_a, mask_a, plen_a)
                cls_b = causal_lm_loss(out_b.logits, ids_b, mask_b, plen_b)
                cls = (cls_a + cls_b) / 2.0

                # [FIX 3] Pass pair_type so positive/negative pairs get different treatment
                con = contrastive_repr_loss(
                    out_a.hidden_states, out_b.hidden_states,
                    mask_a, mask_b, CONTRASTIVE_LAYER_FRAC, pair_type
                )

                # [FIX 2] Normalized loss mixing
                loss = loss_norm.normalize("cls", cls) * 1.0 + LAMBDA_CONTRASTIVE * loss_norm.normalize("con", con)

                bs = ids_a.shape[0]
                m["cls"] += cls.item() * bs
                m["con"] += con.item() * bs
                m["total"] += loss.item() * bs
                m["n"] += bs

            (loss / GRAD_ACCUM).backward()

            if (bi + 1) % GRAD_ACCUM == 0:
                params = list(model.parameters()) + (list(heads.parameters()) if heads else [])
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            if (bi + 1) % 30 == 0:
                n = m["n"]
                parts = [f"cls={m['cls']/n:.4f}"]
                if use_pairs: parts.append(f"con={m['con']/n:.4f}")
                if use_heads: parts.extend([f"vh={m['val_h']/n:.4f}", f"ph={m['plaus_h']/n:.4f}", f"ort={m['ort']/n:.6f}"])
                # [FIX 2] Show EMA scales so you can monitor normalization
                scales = " | scales: " + " ".join(f"{k}={loss_norm.ema[k]:.4f}" for k in loss_keys if loss_norm.initialized[k])
                print(f"  E{epoch+1} [{bi+1}/{len(dataloader)}] {' '.join(parts)}{scales}")

        n = m["n"]
        ep = {"epoch": epoch + 1, "cls": m["cls"]/n, "total": m["total"]/n}
        if use_pairs: ep["contrastive"] = m["con"]/n
        if use_heads: ep.update({"val_head": m["val_h"]/n, "plaus_head": m["plaus_h"]/n, "orth": m["ort"]/n})
        history.append(ep)
        print(f"\n  Epoch {epoch+1}: {ep}\n")

        ckpt = os.path.join(OUTPUT_DIR, f"epoch{epoch+1}")
        model.save_pretrained(ckpt); tokenizer.save_pretrained(ckpt)
        if heads: torch.save(heads.state_dict(), os.path.join(ckpt, "heads.pt"))

    final = os.path.join(OUTPUT_DIR, "final")
    model.save_pretrained(final); tokenizer.save_pretrained(final)
    if heads: torch.save(heads.state_dict(), os.path.join(final, "heads.pt"))
    save_json(history, os.path.join(OUTPUT_DIR, "history.json"))
    print(f"Saved to {final}")
    return model, tokenizer, heads


# ============================================================
# INFERENCE
# ============================================================

def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        logits = model(full_ids, use_cache=False).logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    return lp[0, max(prompt_ids.shape[1] - 1, 0):].sum().item()


@torch.no_grad()
def evaluate(model, tokenizer, test_path, out_path):
    model.eval()
    data = load_json(test_path)
    preds = []
    for i, ex in enumerate(data):
        msgs = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."}
        ]
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        y = score_label(model, tokenizer, prompt, " yes", MAX_LEN)
        n = score_label(model, tokenizer, prompt, " no", MAX_LEN)
        preds.append({"id": ex["id"], "validity": bool(y >= n)})
        if (i + 1) % 50 == 0: print(f"  {i+1}/{len(data)}")
    save_json(preds, out_path)
    print(f"Predictions: {out_path}")
    return preds


def run_eval(pred_path):
    if not os.path.exists(EVAL_SCRIPT):
        print("Eval script not found, skipping official eval")
        return None
    spec = importlib.util.spec_from_file_location("ev", EVAL_SCRIPT)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    out = os.path.join(OUTPUT_DIR, "eval_results.json")
    mod.run_full_scoring(TEST_JSON, pred_path, out)
    r = load_json(out)
    print(f"ACC={r['accuracy']:.2f}  TCE={r['content_effect']:.4f}  Score={r['combined_score']:.4f}")
    return r


def subgroup_acc(test_path, pred_path):
    data = load_json(test_path)
    preds = {p["id"]: p["validity"] for p in load_json(pred_path)}
    buckets = {}
    for ex in data:
        v = "valid" if ex["validity"] else "invalid"
        p = "plausible" if ex["plausibility"] else "implausible"
        k = f"{p}_{v}"
        buckets.setdefault(k, {"c": 0, "t": 0})
        buckets[k]["t"] += 1
        if preds.get(ex["id"]) == ex["validity"]:
            buckets[k]["c"] += 1
    print("\nSubgroup accuracy:")
    for k in sorted(buckets):
        b = buckets[k]
        print(f"  {k:30s}: {100*b['c']/b['t']:6.2f}% ({b['c']}/{b['t']})")


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    model, tokenizer, heads = train()
    pred_path = os.path.join(OUTPUT_DIR, "predictions.json")
    evaluate(model, tokenizer, TEST_JSON, pred_path)
    subgroup_acc(TEST_JSON, pred_path)
    run_eval(pred_path)

Strategy: simple
Model: Qwen/Qwen2.5-3B-Instruct
Pool mode: last_token


Current model requires 512 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 7,372,800 || all params: 3,093,311,488 || trainable%: 0.2383
Dataset: 960 examples, 240 batches/epoch
Total steps: 300



RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cpu and cuda:0! (when checking argument for argument index in method wrapper_CUDA__index_select)

In [5]:
"""
Unified LoRA Fine-Tuning for Syllogistic Reasoning
=====================================================
4 strategies selectable via STRATEGY flag:

  "simple"        = Option 1: validity classification only
  "contrastive"   = Option 2: validity + plausibility-invariance pairs
  "orthogonal"    = Option 3: multi-task with orthogonal separation
  "adversarial"   = Option 4: multi-task with gradient reversal

Fixes applied:
  [1] Single AutoModelForCausalLM for both generation loss and representations.
      We use logits for causal LM loss and hidden_states (pre-lm_head) for
      representation learning. No separate AutoModel needed.
  [2] Loss normalization: each loss component is scale-normalized before
      weighting so no single term dominates.
  [3] Improved contrastive pairing: cross-validity pairs added as negative
      examples (same plausibility, different validity) for harder contrast.
  [4] Weighted attention pooling: last_token still default for decoder,
      but added attention_weighted option using attention scores as weights.
"""

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
import importlib.util

# ============================================================
# CONFIG
# ============================================================
STRATEGY = "contrastive"  # "simple" | "contrastive" | "orthogonal" | "adversarial"

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUTPUT_DIR = f"lora_{STRATEGY}_output"
HF_TOKEN = os.getenv("HF_TOKEN", "")

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]  # reduced from 4 to 2 modules to save ~40% LoRA memory

EPOCHS = 5
BATCH_SIZE = 1            # reduced from 4 — contrastive does 2 forward passes per step
GRAD_ACCUM = 16           # effective batch = 1 * 16 = 16 (same as before)
LR = 2e-4
HEAD_LR = 5e-4
WARMUP_RATIO = 0.1
MAX_LEN = 384             # reduced from 512 — dataset max is 40 words (~60 tokens), 384 is plenty
SEED = 42

# Contrastive
LAMBDA_CONTRASTIVE = 0.5
CONTRASTIVE_LAYER_FRAC = 0.25
USE_NEGATIVE_PAIRS = True  # [FIX 3] add cross-validity negative pairs

# Orthogonal/Adversarial
LAMBDA_PLAUS = 0.3
LAMBDA_ORTH = 0.3          # [FIX 2] reduced from 1.0, gets normalized anyway
LAMBDA_DECORR = 0.1
LAMBDA_VAL_HEAD = 0.3
GRL_LAMBDA = 1.0
HEAD_PROJ_DIM = 256
EXTRACT_LAYER = -1

# [FIX 4] "last_token" | "mean_pool" | "attention_weighted"
POOL_MODE = "last_token"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def log_gpu_mem(tag=""):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"  [GPU {tag}] allocated={alloc:.2f}GB reserved={reserved:.2f}GB")

def load_json(p):
    with open(p, "r", encoding="utf-8") as f: return json.load(f)

def save_json(o, p):
    os.makedirs(os.path.dirname(p) or ".", exist_ok=True)
    with open(p, "w", encoding="utf-8") as f: json.dump(o, f, indent=2, ensure_ascii=False)


# ============================================================
# DATASETS
# ============================================================

class SimpleDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def _prompt(self, syl):
        msgs = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syl}\n\nAnswer yes or no."}
        ]
        return self.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def __getitem__(self, idx):
        ex = self.data[idx]
        prompt = self._prompt(ex["syllogism"])
        target = " yes" if ex["validity"] else " no"
        full = prompt + target
        enc = self.tokenizer(full, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
        penc = self.tokenizer(prompt, truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": enc.input_ids.squeeze(0),
            "attention_mask": enc.attention_mask.squeeze(0),
            "prompt_len": penc.input_ids.shape[1],
            "validity": 1 if ex["validity"] else 0,
            "plausibility": 1 if ex["plausibility"] else 0,
        }


# [FIX 3] Improved pairing with negative examples
class PairDataset(Dataset):
    def __init__(self, data, tokenizer, max_len, use_negatives=True):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.pairs = []

        vp = [x for x in data if x["validity"] and x["plausibility"]]
        vi = [x for x in data if x["validity"] and not x["plausibility"]]
        ip = [x for x in data if not x["validity"] and x["plausibility"]]
        ii = [x for x in data if not x["validity"] and not x["plausibility"]]

        rng = random.Random(SEED)

        # Positive pairs: same validity, different plausibility → push together
        def pos_pair(a, b, val):
            rng.shuffle(a); rng.shuffle(b)
            n = min(len(a), len(b))
            for i in range(n):
                self.pairs.append({
                    "syl_a": a[i]["syllogism"], "syl_b": b[i]["syllogism"],
                    "validity": val, "pair_type": "positive"
                })

        pos_pair(list(vp), list(vi), True)
        pos_pair(list(ip), list(ii), False)

        # Negative pairs: same plausibility, different validity → push apart
        if use_negatives:
            def neg_pair(a, b):
                rng.shuffle(a); rng.shuffle(b)
                n = min(len(a), len(b))
                for i in range(n):
                    self.pairs.append({
                        "syl_a": a[i]["syllogism"], "syl_b": b[i]["syllogism"],
                        "validity_a": a[i]["validity"], "validity_b": b[i]["validity"],
                        "pair_type": "negative"
                    })
            neg_pair(list(vp), list(ip))  # plausible: valid vs invalid
            neg_pair(list(vi), list(ii))  # implausible: valid vs invalid

        rng.shuffle(self.pairs)
        n_pos = sum(1 for p in self.pairs if p["pair_type"] == "positive")
        n_neg = sum(1 for p in self.pairs if p["pair_type"] == "negative")
        print(f"Pairs: {n_pos} positive (push together), {n_neg} negative (push apart)")

    def __len__(self): return len(self.pairs)

    def _prompt(self, syl):
        msgs = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syl}\n\nAnswer yes or no."}
        ]
        return self.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def __getitem__(self, idx):
        p = self.pairs[idx]
        out = {"pair_type": 0 if p["pair_type"] == "positive" else 1}

        if p["pair_type"] == "positive":
            target_a = " yes" if p["validity"] else " no"
            target_b = target_a
        else:
            target_a = " yes" if p["validity_a"] else " no"
            target_b = " yes" if p["validity_b"] else " no"

        for suffix, syl, target in [("_a", p["syl_a"], target_a), ("_b", p["syl_b"], target_b)]:
            prompt = self._prompt(syl)
            full = prompt + target
            enc = self.tokenizer(full, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
            penc = self.tokenizer(prompt, truncation=True, max_length=self.max_len, return_tensors="pt")
            out[f"input_ids{suffix}"] = enc.input_ids.squeeze(0)
            out[f"attention_mask{suffix}"] = enc.attention_mask.squeeze(0)
            out[f"prompt_len{suffix}"] = penc.input_ids.shape[1]

        return out


# ============================================================
# HEADS & GRADIENT REVERSAL
# ============================================================

class GRLFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.clone()
    @staticmethod
    def backward(ctx, grad):
        return -ctx.lam * grad, None

class GRL(nn.Module):
    def __init__(self, lam=1.0): super().__init__(); self.lam = lam
    def forward(self, x): return GRLFunction.apply(x, self.lam)

class DisentangleHeads(nn.Module):
    def __init__(self, hidden_dim, proj_dim, use_grl=False, grl_lam=1.0):
        super().__init__()
        self.val_proj = nn.Sequential(nn.Linear(hidden_dim, proj_dim), nn.ReLU(), nn.Dropout(0.1))
        self.val_cls = nn.Linear(proj_dim, 2)
        self.plaus_proj = nn.Sequential(nn.Linear(hidden_dim, proj_dim), nn.ReLU(), nn.Dropout(0.1))
        self.plaus_cls = nn.Linear(proj_dim, 2)
        self.grl = GRL(grl_lam) if use_grl else nn.Identity()

    def forward(self, h):
        hv = self.val_proj(h)
        hp = self.plaus_proj(self.grl(h))
        return self.val_cls(hv), self.plaus_cls(hp), hv, hp


# ============================================================
# LOSS HELPERS
# ============================================================

def causal_lm_loss(logits, input_ids, attention_mask, prompt_len):
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()
    mask = torch.zeros_like(shift_labels, dtype=torch.float32)
    for i in range(input_ids.shape[0]):
        pl = prompt_len[i].item() if isinstance(prompt_len, torch.Tensor) else prompt_len[i]
        end = attention_mask[i].sum().item() - 1
        mask[i, max(pl - 1, 0):end] = 1.0
    loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), reduction="none").view(shift_labels.shape)
    return (loss * mask).sum() / (mask.sum() + 1e-8)


# [FIX 3] Updated to handle positive (push together) and negative (push apart) pairs
def contrastive_repr_loss(hidden_a, hidden_b, mask_a, mask_b, layer_frac, pair_type):
    n_layers = len(hidden_a)
    start = int(n_layers * (1.0 - layer_frac))
    total = 0.0
    count = 0
    for li in range(start, n_layers):
        ha, hb = hidden_a[li], hidden_b[li]
        pos_a = mask_a.sum(dim=1) - 1
        pos_b = mask_b.sum(dim=1) - 1
        ra = ha[torch.arange(ha.size(0)), pos_a].float()
        rb = hb[torch.arange(hb.size(0)), pos_b].float()
        cos_sim = F.cosine_similarity(ra, rb, dim=-1)

        # Per-example: positive pairs → minimize distance, negative → maximize distance
        # pair_type: 0=positive, 1=negative
        pt = pair_type.float()
        # positive: loss = 1 - cos_sim  (push together)
        # negative: loss = max(0, cos_sim - margin)  (push apart)
        margin = 0.2
        pos_loss = (1.0 - cos_sim) * (1.0 - pt)
        neg_loss = torch.clamp(cos_sim - margin, min=0.0) * pt
        total += (pos_loss + neg_loss).mean()
        count += 1
    return total / max(count, 1)


def orth_loss(hv, hp):
    hv_n = F.normalize(hv, dim=-1)
    hp_n = F.normalize(hp, dim=-1)
    cc = (hv_n.T @ hp_n) / hv.shape[0]
    return torch.sum(cc ** 2)


def decorr_loss(hv, hp):
    def off_diag(h):
        hn = F.normalize(h, dim=0)
        c = hn.T @ hn / h.shape[0]
        m = ~torch.eye(c.shape[0], dtype=torch.bool, device=c.device)
        return torch.sum(c[m] ** 2)
    return off_diag(hv) + off_diag(hp)


# [FIX 4] Pooling with attention_weighted option
def pool_repr(hidden_states, attention_mask, layer_idx=-1, mode="last_token", attentions=None):
    h = hidden_states[layer_idx]  # (batch, seq, hidden)

    if mode == "last_token":
        pos = attention_mask.sum(dim=1) - 1
        return h[torch.arange(h.size(0), device=h.device), pos].float()

    elif mode == "mean_pool":
        mask = attention_mask.unsqueeze(-1).float()
        return ((h * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-8)).float()

    elif mode == "attention_weighted":
        # Use attention weights from last layer as importance scores
        # attentions[-1] shape: (batch, heads, seq, seq)
        if attentions is not None and len(attentions) > 0:
            attn = attentions[-1]  # last layer attention
            # Average across heads, take attention TO the last token
            last_pos = attention_mask.sum(dim=1) - 1  # (batch,)
            # Gather attention weights for last token attending to all positions
            attn_weights = torch.zeros(h.size(0), h.size(1), device=h.device)
            for i in range(h.size(0)):
                lp = last_pos[i].item()
                # Mean across heads: how much the last token attends to each position
                attn_weights[i, :lp+1] = attn[i, :, lp, :lp+1].mean(dim=0)
            # Normalize
            attn_weights = attn_weights * attention_mask.float()
            attn_weights = attn_weights / (attn_weights.sum(dim=1, keepdim=True) + 1e-8)
            # Weighted sum
            return (h * attn_weights.unsqueeze(-1)).sum(dim=1).float()
        else:
            # Fallback to last_token if no attention weights
            pos = attention_mask.sum(dim=1) - 1
            return h[torch.arange(h.size(0), device=h.device), pos].float()

    raise ValueError(f"Unknown pool mode: {mode}")


# [FIX 2] Loss normalizer: tracks running stats to keep losses on similar scales
class LossNormalizer:
    """
    Tracks EMA of each loss component's magnitude.
    Divides each loss by its EMA so all components are ~1.0 before weighting.
    First warmup_steps use simple averaging instead of EMA for stability.
    """
    def __init__(self, keys, momentum=0.9, warmup_steps=50):
        self.momentum = momentum
        self.warmup_steps = warmup_steps
        self.ema = {k: 1.0 for k in keys}
        self.initialized = {k: False for k in keys}
        self.step_count = {k: 0 for k in keys}
        self.warmup_sum = {k: 0.0 for k in keys}

    def normalize(self, key, loss_val):
        v = loss_val.detach().item()
        if v < 1e-8:
            return loss_val
        self.step_count[key] += 1
        if self.step_count[key] <= self.warmup_steps:
            self.warmup_sum[key] += v
            self.ema[key] = self.warmup_sum[key] / self.step_count[key]
            self.initialized[key] = True
        else:
            self.ema[key] = self.momentum * self.ema[key] + (1 - self.momentum) * v
            self.initialized[key] = True
        return loss_val / (self.ema[key] + 1e-8)


# ============================================================
# TRAINING
# ============================================================

def train():
    set_seed(SEED)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    token = HF_TOKEN or None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Strategy: {STRATEGY}")
    print(f"Model: {MODEL_NAME}")
    print(f"Pool mode: {POOL_MODE}")

    # [FIX 1] Single AutoModelForCausalLM for both generation loss and representations.
    # We use logits for causal LM loss and hidden_states (pre-lm_head) for
    # representation learning via the disentanglement heads.
    # No separate AutoModel needed — hidden_states are encoder representations.
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, token=token,
        torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    )
    model = model.to(DEVICE)
    lora_cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
                          lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGET_MODULES, bias="none")
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    # Enable gradient checkpointing: trades compute for ~40% less activation memory
    model.gradient_checkpointing_enable()
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    torch.cuda.empty_cache()
    log_gpu_mem("after model+lora")

    train_data = load_json(TRAIN_JSON)
    use_heads = STRATEGY in ("orthogonal", "adversarial")
    use_pairs = STRATEGY == "contrastive"
    need_attn = (POOL_MODE == "attention_weighted")

    if use_pairs:
        dataset = PairDataset(train_data, tokenizer, MAX_LEN, use_negatives=USE_NEGATIVE_PAIRS)
    else:
        dataset = SimpleDataset(train_data, tokenizer, MAX_LEN)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

    heads = None
    param_groups = [{"params": model.parameters(), "lr": LR, "weight_decay": 0.01}]
    if use_heads:
        heads = DisentangleHeads(
            model.config.hidden_size, HEAD_PROJ_DIM,
            use_grl=(STRATEGY == "adversarial"), grl_lam=GRL_LAMBDA
        ).to(DEVICE)
        param_groups.append({"params": heads.parameters(), "lr": HEAD_LR, "weight_decay": 0.01})
        print(f"Head params: {sum(p.numel() for p in heads.parameters()):,}")

    optimizer = torch.optim.AdamW(param_groups)
    total_steps = (len(dataloader) * EPOCHS) // GRAD_ACCUM
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)

    # [FIX 2] Initialize loss normalizer
    loss_keys = ["cls", "con", "vh", "ph", "ort", "dec"]
    loss_norm = LossNormalizer(loss_keys)

    print(f"Dataset: {len(dataset)} examples, {len(dataloader)} batches/epoch")
    print(f"Total steps: {total_steps}\n")

    model.train()
    if heads: heads.train()
    history = []

    for epoch in range(EPOCHS):
        m = {"cls": 0, "con": 0, "val_h": 0, "plaus_h": 0, "ort": 0, "dec": 0, "total": 0, "n": 0}

        for bi, batch in enumerate(dataloader):
            # ---- SIMPLE / ORTHOGONAL / ADVERSARIAL ----
            if not use_pairs:
                ids = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                plen = batch["prompt_len"]
                val_lbl = batch["validity"].to(DEVICE)
                plaus_lbl = batch["plausibility"].to(DEVICE)

                out = model(ids, attention_mask=mask,
                            output_hidden_states=use_heads,
                            output_attentions=need_attn,
                            use_cache=False)

                cls = causal_lm_loss(out.logits, ids, mask, plen)
                # [FIX 2] Normalize before weighting
                loss = loss_norm.normalize("cls", cls) * 1.0
                m["cls"] += cls.item() * ids.shape[0]

                if use_heads:
                    attns = out.attentions if need_attn else None
                    rep = pool_repr(out.hidden_states, mask, EXTRACT_LAYER, POOL_MODE, attns)
                    vl, pl, hv, hp = heads(rep)
                    vh_loss = F.cross_entropy(vl, val_lbl)
                    ph_loss = F.cross_entropy(pl, plaus_lbl)
                    ol = orth_loss(hv, hp)
                    dl = decorr_loss(hv, hp)

                    # [FIX 2] Each component normalized then weighted
                    loss = loss + LAMBDA_VAL_HEAD * loss_norm.normalize("vh", vh_loss)
                    loss = loss + LAMBDA_PLAUS * loss_norm.normalize("ph", ph_loss)
                    loss = loss + LAMBDA_ORTH * loss_norm.normalize("ort", ol)
                    loss = loss + LAMBDA_DECORR * loss_norm.normalize("dec", dl)

                    m["val_h"] += vh_loss.item() * ids.shape[0]
                    m["plaus_h"] += ph_loss.item() * ids.shape[0]
                    m["ort"] += ol.item() * ids.shape[0]
                    m["dec"] += dl.item() * ids.shape[0]

                m["total"] += loss.item() * ids.shape[0]
                m["n"] += ids.shape[0]

            # ---- CONTRASTIVE ----
            else:
                ids_a = batch["input_ids_a"].to(DEVICE)
                mask_a = batch["attention_mask_a"].to(DEVICE)
                plen_a = batch["prompt_len_a"]
                ids_b = batch["input_ids_b"].to(DEVICE)
                mask_b = batch["attention_mask_b"].to(DEVICE)
                plen_b = batch["prompt_len_b"]
                pair_type = batch["pair_type"].to(DEVICE)

                # Sequential forward passes to halve peak memory.
                # Pass A: compute CLS loss with grad, cache hidden states for contrastive loss
                out_a = model(ids_a, attention_mask=mask_a, output_hidden_states=True, use_cache=False)
                cls_a = causal_lm_loss(out_a.logits, ids_a, mask_a, plen_a)
                # Detach hidden states — no grad through them for contrastive loss
                hidden_a = tuple(h.detach() for h in out_a.hidden_states)
                del out_a
                torch.cuda.empty_cache()

                # Pass B: compute CLS loss with grad, cache hidden states
                out_b = model(ids_b, attention_mask=mask_b, output_hidden_states=True, use_cache=False)
                cls_b = causal_lm_loss(out_b.logits, ids_b, mask_b, plen_b)
                hidden_b = tuple(h.detach() for h in out_b.hidden_states)
                del out_b
                torch.cuda.empty_cache()

                cls = (cls_a + cls_b) / 2.0

                # Contrastive loss on detached hiddens (no second-order grad needed)
                con = contrastive_repr_loss(
                    hidden_a, hidden_b,
                    mask_a, mask_b, CONTRASTIVE_LAYER_FRAC, pair_type
                )
                del hidden_a, hidden_b

                # [FIX 2] Normalized loss mixing
                loss = loss_norm.normalize("cls", cls) * 1.0 + LAMBDA_CONTRASTIVE * loss_norm.normalize("con", con)

                bs = ids_a.shape[0]
                m["cls"] += cls.item() * bs
                m["con"] += con.item() * bs
                m["total"] += loss.item() * bs
                m["n"] += bs

            (loss / GRAD_ACCUM).backward()

            if (bi + 1) % GRAD_ACCUM == 0:
                params = list(model.parameters()) + (list(heads.parameters()) if heads else [])
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            if (bi + 1) % 30 == 0:
                n = m["n"]
                parts = [f"cls={m['cls']/n:.4f}"]
                if use_pairs: parts.append(f"con={m['con']/n:.4f}")
                if use_heads: parts.extend([f"vh={m['val_h']/n:.4f}", f"ph={m['plaus_h']/n:.4f}", f"ort={m['ort']/n:.6f}"])
                # [FIX 2] Show EMA scales so you can monitor normalization
                scales = " | scales: " + " ".join(f"{k}={loss_norm.ema[k]:.4f}" for k in loss_keys if loss_norm.initialized[k])
                print(f"  E{epoch+1} [{bi+1}/{len(dataloader)}] {' '.join(parts)}{scales}")

        n = m["n"]
        ep = {"epoch": epoch + 1, "cls": m["cls"]/n, "total": m["total"]/n}
        if use_pairs: ep["contrastive"] = m["con"]/n
        if use_heads: ep.update({"val_head": m["val_h"]/n, "plaus_head": m["plaus_h"]/n, "orth": m["ort"]/n})
        history.append(ep)
        print(f"\n  Epoch {epoch+1}: {ep}\n")

        ckpt = os.path.join(OUTPUT_DIR, f"epoch{epoch+1}")
        model.save_pretrained(ckpt); tokenizer.save_pretrained(ckpt)
        if heads: torch.save(heads.state_dict(), os.path.join(ckpt, "heads.pt"))

    final = os.path.join(OUTPUT_DIR, "final")
    model.save_pretrained(final); tokenizer.save_pretrained(final)
    if heads: torch.save(heads.state_dict(), os.path.join(final, "heads.pt"))
    save_json(history, os.path.join(OUTPUT_DIR, "history.json"))
    print(f"Saved to {final}")
    return model, tokenizer, heads


# ============================================================
# INFERENCE
# ============================================================

def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        logits = model(full_ids, use_cache=False).logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    return lp[0, max(prompt_ids.shape[1] - 1, 0):].sum().item()


@torch.no_grad()
def evaluate(model, tokenizer, test_path, out_path):
    model.eval()
    data = load_json(test_path)
    preds = []
    for i, ex in enumerate(data):
        msgs = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."}
        ]
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        y = score_label(model, tokenizer, prompt, " yes", MAX_LEN)
        n = score_label(model, tokenizer, prompt, " no", MAX_LEN)
        preds.append({"id": ex["id"], "validity": bool(y >= n)})
        if (i + 1) % 50 == 0: print(f"  {i+1}/{len(data)}")
    save_json(preds, out_path)
    print(f"Predictions: {out_path}")
    return preds


def run_eval(pred_path):
    if not os.path.exists(EVAL_SCRIPT):
        print("Eval script not found, skipping official eval")
        return None
    spec = importlib.util.spec_from_file_location("ev", EVAL_SCRIPT)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    out = os.path.join(OUTPUT_DIR, "eval_results.json")
    mod.run_full_scoring(TEST_JSON, pred_path, out)
    r = load_json(out)
    print(f"ACC={r['accuracy']:.2f}  TCE={r['content_effect']:.4f}  Score={r['combined_score']:.4f}")
    return r


def subgroup_acc(test_path, pred_path):
    data = load_json(test_path)
    preds = {p["id"]: p["validity"] for p in load_json(pred_path)}
    buckets = {}
    for ex in data:
        v = "valid" if ex["validity"] else "invalid"
        p = "plausible" if ex["plausibility"] else "implausible"
        k = f"{p}_{v}"
        buckets.setdefault(k, {"c": 0, "t": 0})
        buckets[k]["t"] += 1
        if preds.get(ex["id"]) == ex["validity"]:
            buckets[k]["c"] += 1
    print("\nSubgroup accuracy:")
    for k in sorted(buckets):
        b = buckets[k]
        print(f"  {k:30s}: {100*b['c']/b['t']:6.2f}% ({b['c']}/{b['t']})")


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    model, tokenizer, heads = train()
    pred_path = os.path.join(OUTPUT_DIR, "predictions.json")
    evaluate(model, tokenizer, TEST_JSON, pred_path)
    subgroup_acc(TEST_JSON, pred_path)
    run_eval(pred_path)

Strategy: contrastive
Model: Qwen/Qwen2.5-3B-Instruct
Pool mode: last_token


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193
  [GPU after model+lora] allocated=5.88GB reserved=11.62GB
Pairs: 474 positive (push together), 474 negative (push apart)
Dataset: 948 examples, 948 batches/epoch
Total steps: 296



/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


  E1 [30/948] cls=17.7531 con=0.3802 | scales: cls=17.7531 con=0.3802
  E1 [60/948] cls=18.5651 con=0.3936 | scales: cls=18.5349 con=0.3515
  E1 [90/948] cls=18.3076 con=0.4060 | scales: cls=17.7951 con=0.4826
  E1 [120/948] cls=18.1516 con=0.3814 | scales: cls=17.1582 con=0.2909
  E1 [150/948] cls=17.8733 con=0.3911 | scales: cls=17.1881 con=0.4988
  E1 [180/948] cls=17.6095 con=0.3809 | scales: cls=17.5921 con=0.3616
  E1 [210/948] cls=17.1095 con=0.3808 | scales: cls=13.9745 con=0.3526
  E1 [240/948] cls=16.5131 con=0.3994 | scales: cls=11.5062 con=0.4867
  E1 [270/948] cls=15.6570 con=0.3974 | scales: cls=8.8735 con=0.3592
  E1 [300/948] cls=14.7080 con=0.3930 | scales: cls=5.6382 con=0.4306
  E1 [330/948] cls=13.6662 con=0.3940 | scales: cls=3.2074 con=0.3882
  E1 [360/948] cls=12.7289 con=0.3925 | scales: cls=2.3053 con=0.3345
  E1 [390/948] cls=11.8276 con=0.3895 | scales: cls=0.8868 con=0.4235
  E1 [420/948] cls=11.0814 con=0.3870 | scales: cls=1.2913 con=0.3118
  E1 [450/948] 

In [6]:
"""
Unified LoRA Fine-Tuning for Syllogistic Reasoning
=====================================================
4 strategies selectable via STRATEGY flag:

  "simple"        = Option 1: validity classification only
  "contrastive"   = Option 2: validity + plausibility-invariance pairs
  "orthogonal"    = Option 3: multi-task with orthogonal separation
  "adversarial"   = Option 4: multi-task with gradient reversal

Fixes applied:
  [1] Single AutoModelForCausalLM for both generation loss and representations.
      We use logits for causal LM loss and hidden_states (pre-lm_head) for
      representation learning. No separate AutoModel needed.
  [2] Loss normalization: each loss component is scale-normalized before
      weighting so no single term dominates.
  [3] Improved contrastive pairing: cross-validity pairs added as negative
      examples (same plausibility, different validity) for harder contrast.
  [4] Weighted attention pooling: last_token still default for decoder,
      but added attention_weighted option using attention scores as weights.
"""

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
import importlib.util

# ============================================================
# CONFIG
# ============================================================
STRATEGY = "orthogonal"  # "simple" | "contrastive" | "orthogonal" | "adversarial"

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUTPUT_DIR = f"lora_{STRATEGY}_output"
HF_TOKEN = os.getenv("HF_TOKEN", "")

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]  # reduced from 4 to 2 modules to save ~40% LoRA memory

EPOCHS = 4
BATCH_SIZE = 1            # reduced from 4 — contrastive does 2 forward passes per step
GRAD_ACCUM = 16           # effective batch = 1 * 16 = 16 (same as before)
LR = 2e-4
HEAD_LR = 5e-4
WARMUP_RATIO = 0.1
MAX_LEN = 384             # reduced from 512 — dataset max is 40 words (~60 tokens), 384 is plenty
SEED = 42

# Contrastive
LAMBDA_CONTRASTIVE = 0.5
CONTRASTIVE_LAYER_FRAC = 0.25
USE_NEGATIVE_PAIRS = True  # [FIX 3] add cross-validity negative pairs

# Orthogonal/Adversarial
LAMBDA_PLAUS = 0.3
LAMBDA_ORTH = 0.3          # [FIX 2] reduced from 1.0, gets normalized anyway
LAMBDA_DECORR = 0.1
LAMBDA_VAL_HEAD = 0.3
GRL_LAMBDA = 1.0
HEAD_PROJ_DIM = 256
EXTRACT_LAYER = -1

# [FIX 4] "last_token" | "mean_pool" | "attention_weighted"
POOL_MODE = "last_token"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def log_gpu_mem(tag=""):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"  [GPU {tag}] allocated={alloc:.2f}GB reserved={reserved:.2f}GB")

def load_json(p):
    with open(p, "r", encoding="utf-8") as f: return json.load(f)

def save_json(o, p):
    os.makedirs(os.path.dirname(p) or ".", exist_ok=True)
    with open(p, "w", encoding="utf-8") as f: json.dump(o, f, indent=2, ensure_ascii=False)


# ============================================================
# DATASETS
# ============================================================

class SimpleDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def _prompt(self, syl):
        msgs = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syl}\n\nAnswer yes or no."}
        ]
        return self.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def __getitem__(self, idx):
        ex = self.data[idx]
        prompt = self._prompt(ex["syllogism"])
        target = " yes" if ex["validity"] else " no"
        full = prompt + target
        enc = self.tokenizer(full, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
        penc = self.tokenizer(prompt, truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": enc.input_ids.squeeze(0),
            "attention_mask": enc.attention_mask.squeeze(0),
            "prompt_len": penc.input_ids.shape[1],
            "validity": 1 if ex["validity"] else 0,
            "plausibility": 1 if ex["plausibility"] else 0,
        }


# [FIX 3] Improved pairing with negative examples
class PairDataset(Dataset):
    def __init__(self, data, tokenizer, max_len, use_negatives=True):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.pairs = []

        vp = [x for x in data if x["validity"] and x["plausibility"]]
        vi = [x for x in data if x["validity"] and not x["plausibility"]]
        ip = [x for x in data if not x["validity"] and x["plausibility"]]
        ii = [x for x in data if not x["validity"] and not x["plausibility"]]

        rng = random.Random(SEED)

        # Positive pairs: same validity, different plausibility → push together
        def pos_pair(a, b, val):
            rng.shuffle(a); rng.shuffle(b)
            n = min(len(a), len(b))
            for i in range(n):
                self.pairs.append({
                    "syl_a": a[i]["syllogism"], "syl_b": b[i]["syllogism"],
                    "validity": val, "pair_type": "positive"
                })

        pos_pair(list(vp), list(vi), True)
        pos_pair(list(ip), list(ii), False)

        # Negative pairs: same plausibility, different validity → push apart
        if use_negatives:
            def neg_pair(a, b):
                rng.shuffle(a); rng.shuffle(b)
                n = min(len(a), len(b))
                for i in range(n):
                    self.pairs.append({
                        "syl_a": a[i]["syllogism"], "syl_b": b[i]["syllogism"],
                        "validity_a": a[i]["validity"], "validity_b": b[i]["validity"],
                        "pair_type": "negative"
                    })
            neg_pair(list(vp), list(ip))  # plausible: valid vs invalid
            neg_pair(list(vi), list(ii))  # implausible: valid vs invalid

        rng.shuffle(self.pairs)
        n_pos = sum(1 for p in self.pairs if p["pair_type"] == "positive")
        n_neg = sum(1 for p in self.pairs if p["pair_type"] == "negative")
        print(f"Pairs: {n_pos} positive (push together), {n_neg} negative (push apart)")

    def __len__(self): return len(self.pairs)

    def _prompt(self, syl):
        msgs = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syl}\n\nAnswer yes or no."}
        ]
        return self.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def __getitem__(self, idx):
        p = self.pairs[idx]
        out = {"pair_type": 0 if p["pair_type"] == "positive" else 1}

        if p["pair_type"] == "positive":
            target_a = " yes" if p["validity"] else " no"
            target_b = target_a
        else:
            target_a = " yes" if p["validity_a"] else " no"
            target_b = " yes" if p["validity_b"] else " no"

        for suffix, syl, target in [("_a", p["syl_a"], target_a), ("_b", p["syl_b"], target_b)]:
            prompt = self._prompt(syl)
            full = prompt + target
            enc = self.tokenizer(full, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
            penc = self.tokenizer(prompt, truncation=True, max_length=self.max_len, return_tensors="pt")
            out[f"input_ids{suffix}"] = enc.input_ids.squeeze(0)
            out[f"attention_mask{suffix}"] = enc.attention_mask.squeeze(0)
            out[f"prompt_len{suffix}"] = penc.input_ids.shape[1]

        return out


# ============================================================
# HEADS & GRADIENT REVERSAL
# ============================================================

class GRLFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.clone()
    @staticmethod
    def backward(ctx, grad):
        return -ctx.lam * grad, None

class GRL(nn.Module):
    def __init__(self, lam=1.0): super().__init__(); self.lam = lam
    def forward(self, x): return GRLFunction.apply(x, self.lam)

class DisentangleHeads(nn.Module):
    def __init__(self, hidden_dim, proj_dim, use_grl=False, grl_lam=1.0):
        super().__init__()
        self.val_proj = nn.Sequential(nn.Linear(hidden_dim, proj_dim), nn.ReLU(), nn.Dropout(0.1))
        self.val_cls = nn.Linear(proj_dim, 2)
        self.plaus_proj = nn.Sequential(nn.Linear(hidden_dim, proj_dim), nn.ReLU(), nn.Dropout(0.1))
        self.plaus_cls = nn.Linear(proj_dim, 2)
        self.grl = GRL(grl_lam) if use_grl else nn.Identity()

    def forward(self, h):
        hv = self.val_proj(h)
        hp = self.plaus_proj(self.grl(h))
        return self.val_cls(hv), self.plaus_cls(hp), hv, hp


# ============================================================
# LOSS HELPERS
# ============================================================

def causal_lm_loss(logits, input_ids, attention_mask, prompt_len):
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()
    mask = torch.zeros_like(shift_labels, dtype=torch.float32)
    for i in range(input_ids.shape[0]):
        pl = prompt_len[i].item() if isinstance(prompt_len, torch.Tensor) else prompt_len[i]
        end = attention_mask[i].sum().item() - 1
        mask[i, max(pl - 1, 0):end] = 1.0
    loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), reduction="none").view(shift_labels.shape)
    return (loss * mask).sum() / (mask.sum() + 1e-8)


# [FIX 3] Updated to handle positive (push together) and negative (push apart) pairs
def contrastive_repr_loss(hidden_a, hidden_b, mask_a, mask_b, layer_frac, pair_type):
    n_layers = len(hidden_a)
    start = int(n_layers * (1.0 - layer_frac))
    total = 0.0
    count = 0
    for li in range(start, n_layers):
        ha, hb = hidden_a[li], hidden_b[li]
        pos_a = mask_a.sum(dim=1) - 1
        pos_b = mask_b.sum(dim=1) - 1
        ra = ha[torch.arange(ha.size(0)), pos_a].float()
        rb = hb[torch.arange(hb.size(0)), pos_b].float()
        cos_sim = F.cosine_similarity(ra, rb, dim=-1)

        # Per-example: positive pairs → minimize distance, negative → maximize distance
        # pair_type: 0=positive, 1=negative
        pt = pair_type.float()
        # positive: loss = 1 - cos_sim  (push together)
        # negative: loss = max(0, cos_sim - margin)  (push apart)
        margin = 0.2
        pos_loss = (1.0 - cos_sim) * (1.0 - pt)
        neg_loss = torch.clamp(cos_sim - margin, min=0.0) * pt
        total += (pos_loss + neg_loss).mean()
        count += 1
    return total / max(count, 1)


def orth_loss(hv, hp):
    hv_n = F.normalize(hv, dim=-1)
    hp_n = F.normalize(hp, dim=-1)
    cc = (hv_n.T @ hp_n) / hv.shape[0]
    return torch.sum(cc ** 2)


def decorr_loss(hv, hp):
    def off_diag(h):
        hn = F.normalize(h, dim=0)
        c = hn.T @ hn / h.shape[0]
        m = ~torch.eye(c.shape[0], dtype=torch.bool, device=c.device)
        return torch.sum(c[m] ** 2)
    return off_diag(hv) + off_diag(hp)


# [FIX 4] Pooling with attention_weighted option
def pool_repr(hidden_states, attention_mask, layer_idx=-1, mode="last_token", attentions=None):
    h = hidden_states[layer_idx]  # (batch, seq, hidden)

    if mode == "last_token":
        pos = attention_mask.sum(dim=1) - 1
        return h[torch.arange(h.size(0), device=h.device), pos].float()

    elif mode == "mean_pool":
        mask = attention_mask.unsqueeze(-1).float()
        return ((h * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-8)).float()

    elif mode == "attention_weighted":
        # Use attention weights from last layer as importance scores
        # attentions[-1] shape: (batch, heads, seq, seq)
        if attentions is not None and len(attentions) > 0:
            attn = attentions[-1]  # last layer attention
            # Average across heads, take attention TO the last token
            last_pos = attention_mask.sum(dim=1) - 1  # (batch,)
            # Gather attention weights for last token attending to all positions
            attn_weights = torch.zeros(h.size(0), h.size(1), device=h.device)
            for i in range(h.size(0)):
                lp = last_pos[i].item()
                # Mean across heads: how much the last token attends to each position
                attn_weights[i, :lp+1] = attn[i, :, lp, :lp+1].mean(dim=0)
            # Normalize
            attn_weights = attn_weights * attention_mask.float()
            attn_weights = attn_weights / (attn_weights.sum(dim=1, keepdim=True) + 1e-8)
            # Weighted sum
            return (h * attn_weights.unsqueeze(-1)).sum(dim=1).float()
        else:
            # Fallback to last_token if no attention weights
            pos = attention_mask.sum(dim=1) - 1
            return h[torch.arange(h.size(0), device=h.device), pos].float()

    raise ValueError(f"Unknown pool mode: {mode}")


# [FIX 2] Loss normalizer: tracks running stats to keep losses on similar scales
class LossNormalizer:
    """
    Tracks EMA of each loss component's magnitude.
    Divides each loss by its EMA so all components are ~1.0 before weighting.
    First warmup_steps use simple averaging instead of EMA for stability.
    """
    def __init__(self, keys, momentum=0.9, warmup_steps=50):
        self.momentum = momentum
        self.warmup_steps = warmup_steps
        self.ema = {k: 1.0 for k in keys}
        self.initialized = {k: False for k in keys}
        self.step_count = {k: 0 for k in keys}
        self.warmup_sum = {k: 0.0 for k in keys}

    def normalize(self, key, loss_val):
        v = loss_val.detach().item()
        if v < 1e-8:
            return loss_val
        self.step_count[key] += 1
        if self.step_count[key] <= self.warmup_steps:
            self.warmup_sum[key] += v
            self.ema[key] = self.warmup_sum[key] / self.step_count[key]
            self.initialized[key] = True
        else:
            self.ema[key] = self.momentum * self.ema[key] + (1 - self.momentum) * v
            self.initialized[key] = True
        return loss_val / (self.ema[key] + 1e-8)


# ============================================================
# TRAINING
# ============================================================

def train():
    set_seed(SEED)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    token = HF_TOKEN or None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Strategy: {STRATEGY}")
    print(f"Model: {MODEL_NAME}")
    print(f"Pool mode: {POOL_MODE}")

    # [FIX 1] Single AutoModelForCausalLM for both generation loss and representations.
    # We use logits for causal LM loss and hidden_states (pre-lm_head) for
    # representation learning via the disentanglement heads.
    # No separate AutoModel needed — hidden_states are encoder representations.
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, token=token,
        torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    )
    model = model.to(DEVICE)
    lora_cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
                          lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGET_MODULES, bias="none")
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    # Enable gradient checkpointing: trades compute for ~40% less activation memory
    model.gradient_checkpointing_enable()
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    torch.cuda.empty_cache()
    log_gpu_mem("after model+lora")

    train_data = load_json(TRAIN_JSON)
    use_heads = STRATEGY in ("orthogonal", "adversarial")
    use_pairs = STRATEGY == "contrastive"
    need_attn = (POOL_MODE == "attention_weighted")

    if use_pairs:
        dataset = PairDataset(train_data, tokenizer, MAX_LEN, use_negatives=USE_NEGATIVE_PAIRS)
    else:
        dataset = SimpleDataset(train_data, tokenizer, MAX_LEN)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

    heads = None
    param_groups = [{"params": model.parameters(), "lr": LR, "weight_decay": 0.01}]
    if use_heads:
        heads = DisentangleHeads(
            model.config.hidden_size, HEAD_PROJ_DIM,
            use_grl=(STRATEGY == "adversarial"), grl_lam=GRL_LAMBDA
        ).to(DEVICE)
        param_groups.append({"params": heads.parameters(), "lr": HEAD_LR, "weight_decay": 0.01})
        print(f"Head params: {sum(p.numel() for p in heads.parameters()):,}")

    optimizer = torch.optim.AdamW(param_groups)
    total_steps = (len(dataloader) * EPOCHS) // GRAD_ACCUM
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)

    # [FIX 2] Initialize loss normalizer
    loss_keys = ["cls", "con", "vh", "ph", "ort", "dec"]
    loss_norm = LossNormalizer(loss_keys)

    print(f"Dataset: {len(dataset)} examples, {len(dataloader)} batches/epoch")
    print(f"Total steps: {total_steps}\n")

    model.train()
    if heads: heads.train()
    history = []

    for epoch in range(EPOCHS):
        m = {"cls": 0, "con": 0, "val_h": 0, "plaus_h": 0, "ort": 0, "dec": 0, "total": 0, "n": 0}

        for bi, batch in enumerate(dataloader):
            # ---- SIMPLE / ORTHOGONAL / ADVERSARIAL ----
            if not use_pairs:
                ids = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                plen = batch["prompt_len"]
                val_lbl = batch["validity"].to(DEVICE)
                plaus_lbl = batch["plausibility"].to(DEVICE)

                out = model(ids, attention_mask=mask,
                            output_hidden_states=use_heads,
                            output_attentions=need_attn,
                            use_cache=False)

                cls = causal_lm_loss(out.logits, ids, mask, plen)
                # [FIX 2] Normalize before weighting
                loss = loss_norm.normalize("cls", cls) * 1.0
                m["cls"] += cls.item() * ids.shape[0]

                if use_heads:
                    attns = out.attentions if need_attn else None
                    rep = pool_repr(out.hidden_states, mask, EXTRACT_LAYER, POOL_MODE, attns)
                    vl, pl, hv, hp = heads(rep)
                    vh_loss = F.cross_entropy(vl, val_lbl)
                    ph_loss = F.cross_entropy(pl, plaus_lbl)
                    ol = orth_loss(hv, hp)
                    dl = decorr_loss(hv, hp)

                    # [FIX 2] Each component normalized then weighted
                    loss = loss + LAMBDA_VAL_HEAD * loss_norm.normalize("vh", vh_loss)
                    loss = loss + LAMBDA_PLAUS * loss_norm.normalize("ph", ph_loss)
                    loss = loss + LAMBDA_ORTH * loss_norm.normalize("ort", ol)
                    loss = loss + LAMBDA_DECORR * loss_norm.normalize("dec", dl)

                    m["val_h"] += vh_loss.item() * ids.shape[0]
                    m["plaus_h"] += ph_loss.item() * ids.shape[0]
                    m["ort"] += ol.item() * ids.shape[0]
                    m["dec"] += dl.item() * ids.shape[0]

                m["total"] += loss.item() * ids.shape[0]
                m["n"] += ids.shape[0]

            # ---- CONTRASTIVE ----
            else:
                ids_a = batch["input_ids_a"].to(DEVICE)
                mask_a = batch["attention_mask_a"].to(DEVICE)
                plen_a = batch["prompt_len_a"]
                ids_b = batch["input_ids_b"].to(DEVICE)
                mask_b = batch["attention_mask_b"].to(DEVICE)
                plen_b = batch["prompt_len_b"]
                pair_type = batch["pair_type"].to(DEVICE)

                # Sequential forward passes to halve peak memory.
                # Pass A: compute CLS loss with grad, cache hidden states for contrastive loss
                out_a = model(ids_a, attention_mask=mask_a, output_hidden_states=True, use_cache=False)
                cls_a = causal_lm_loss(out_a.logits, ids_a, mask_a, plen_a)
                # Detach hidden states — no grad through them for contrastive loss
                hidden_a = tuple(h.detach() for h in out_a.hidden_states)
                del out_a
                torch.cuda.empty_cache()

                # Pass B: compute CLS loss with grad, cache hidden states
                out_b = model(ids_b, attention_mask=mask_b, output_hidden_states=True, use_cache=False)
                cls_b = causal_lm_loss(out_b.logits, ids_b, mask_b, plen_b)
                hidden_b = tuple(h.detach() for h in out_b.hidden_states)
                del out_b
                torch.cuda.empty_cache()

                cls = (cls_a + cls_b) / 2.0

                # Contrastive loss on detached hiddens (no second-order grad needed)
                con = contrastive_repr_loss(
                    hidden_a, hidden_b,
                    mask_a, mask_b, CONTRASTIVE_LAYER_FRAC, pair_type
                )
                del hidden_a, hidden_b

                # [FIX 2] Normalized loss mixing
                loss = loss_norm.normalize("cls", cls) * 1.0 + LAMBDA_CONTRASTIVE * loss_norm.normalize("con", con)

                bs = ids_a.shape[0]
                m["cls"] += cls.item() * bs
                m["con"] += con.item() * bs
                m["total"] += loss.item() * bs
                m["n"] += bs

            (loss / GRAD_ACCUM).backward()

            if (bi + 1) % GRAD_ACCUM == 0:
                params = list(model.parameters()) + (list(heads.parameters()) if heads else [])
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            if (bi + 1) % 30 == 0:
                n = m["n"]
                parts = [f"cls={m['cls']/n:.4f}"]
                if use_pairs: parts.append(f"con={m['con']/n:.4f}")
                if use_heads: parts.extend([f"vh={m['val_h']/n:.4f}", f"ph={m['plaus_h']/n:.4f}", f"ort={m['ort']/n:.6f}"])
                # [FIX 2] Show EMA scales so you can monitor normalization
                scales = " | scales: " + " ".join(f"{k}={loss_norm.ema[k]:.4f}" for k in loss_keys if loss_norm.initialized[k])
                print(f"  E{epoch+1} [{bi+1}/{len(dataloader)}] {' '.join(parts)}{scales}")

        n = m["n"]
        ep = {"epoch": epoch + 1, "cls": m["cls"]/n, "total": m["total"]/n}
        if use_pairs: ep["contrastive"] = m["con"]/n
        if use_heads: ep.update({"val_head": m["val_h"]/n, "plaus_head": m["plaus_h"]/n, "orth": m["ort"]/n})
        history.append(ep)
        print(f"\n  Epoch {epoch+1}: {ep}\n")

        ckpt = os.path.join(OUTPUT_DIR, f"epoch{epoch+1}")
        model.save_pretrained(ckpt); tokenizer.save_pretrained(ckpt)
        if heads: torch.save(heads.state_dict(), os.path.join(ckpt, "heads.pt"))

    final = os.path.join(OUTPUT_DIR, "final")
    model.save_pretrained(final); tokenizer.save_pretrained(final)
    if heads: torch.save(heads.state_dict(), os.path.join(final, "heads.pt"))
    save_json(history, os.path.join(OUTPUT_DIR, "history.json"))
    print(f"Saved to {final}")
    return model, tokenizer, heads


# ============================================================
# INFERENCE
# ============================================================

def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        logits = model(full_ids, use_cache=False).logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    return lp[0, max(prompt_ids.shape[1] - 1, 0):].sum().item()


@torch.no_grad()
def evaluate(model, tokenizer, test_path, out_path):
    model.eval()
    data = load_json(test_path)
    preds = []
    for i, ex in enumerate(data):
        msgs = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."}
        ]
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        y = score_label(model, tokenizer, prompt, " yes", MAX_LEN)
        n = score_label(model, tokenizer, prompt, " no", MAX_LEN)
        preds.append({"id": ex["id"], "validity": bool(y >= n)})
        if (i + 1) % 50 == 0: print(f"  {i+1}/{len(data)}")
    save_json(preds, out_path)
    print(f"Predictions: {out_path}")
    return preds


def run_eval(pred_path):
    if not os.path.exists(EVAL_SCRIPT):
        print("Eval script not found, skipping official eval")
        return None
    spec = importlib.util.spec_from_file_location("ev", EVAL_SCRIPT)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    out = os.path.join(OUTPUT_DIR, "eval_results.json")
    mod.run_full_scoring(TEST_JSON, pred_path, out)
    r = load_json(out)
    print(f"ACC={r['accuracy']:.2f}  TCE={r['content_effect']:.4f}  Score={r['combined_score']:.4f}")
    return r


def subgroup_acc(test_path, pred_path):
    data = load_json(test_path)
    preds = {p["id"]: p["validity"] for p in load_json(pred_path)}
    buckets = {}
    for ex in data:
        v = "valid" if ex["validity"] else "invalid"
        p = "plausible" if ex["plausibility"] else "implausible"
        k = f"{p}_{v}"
        buckets.setdefault(k, {"c": 0, "t": 0})
        buckets[k]["t"] += 1
        if preds.get(ex["id"]) == ex["validity"]:
            buckets[k]["c"] += 1
    print("\nSubgroup accuracy:")
    for k in sorted(buckets):
        b = buckets[k]
        print(f"  {k:30s}: {100*b['c']/b['t']:6.2f}% ({b['c']}/{b['t']})")


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    model, tokenizer, heads = train()
    pred_path = os.path.join(OUTPUT_DIR, "predictions.json")
    evaluate(model, tokenizer, TEST_JSON, pred_path)
    subgroup_acc(TEST_JSON, pred_path)
    run_eval(pred_path)

Strategy: orthogonal
Model: Qwen/Qwen2.5-3B-Instruct
Pool mode: last_token


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193
  [GPU after model+lora] allocated=11.66GB reserved=11.71GB
Head params: 1,050,116
Dataset: 960 examples, 960 batches/epoch
Total steps: 240

  E1 [30/960] cls=18.2417 vh=0.8406 ph=0.7090 ort=1.000000 | scales: cls=18.2417 vh=0.8406 ph=0.7090 ort=1.0000 dec=29487.6000
  E1 [60/960] cls=18.0271 vh=0.7891 ph=0.6749 ort=1.000000 | scales: cls=17.5459 vh=0.8272 ph=0.6846 ort=1.0000 dec=28744.5108
  E1 [90/960] cls=18.8340 vh=0.8646 ph=0.7231 ort=1.000000 | scales: cls=20.6474 vh=1.0794 ph=0.8193 ort=1.0000 dec=27256.4197
  E1 [120/960] cls=18.3854 vh=0.8503 ph=0.7245 ort=1.000000 | scales: cls=16.8993 vh=0.7220 ph=0.7932 ort=1.0000 dec=26492.2506
  E1 [150/960] cls=18.0596 vh=0.7838 ph=0.7253 ort=1.000000 | scales: cls=17.1632 vh=0.4971 ph=0.7448 ort=1.0000 dec=26294.3490
  E1 [180/960] cls=17.6760 vh=0.7482 ph=0.7293 ort=1.000000 | scales: cls=15.4638 vh=0.3956 ph=0.7355 ort=1.0000 dec=25998.9576
  E1 [210/960]

In [9]:
"""
Unified LoRA Fine-Tuning for Syllogistic Reasoning
=====================================================
4 strategies selectable via STRATEGY flag:

  "simple"        = Option 1: validity classification only
  "contrastive"   = Option 2: validity + plausibility-invariance pairs
  "orthogonal"    = Option 3: multi-task with orthogonal separation
  "adversarial"   = Option 4: multi-task with gradient reversal

Fixes applied:
  [1] Single AutoModelForCausalLM for both generation loss and representations.
      We use logits for causal LM loss and hidden_states (pre-lm_head) for
      representation learning. No separate AutoModel needed.
  [2] Loss normalization: each loss component is scale-normalized before
      weighting so no single term dominates.
  [3] Improved contrastive pairing: cross-validity pairs added as negative
      examples (same plausibility, different validity) for harder contrast.
  [4] Weighted attention pooling: last_token still default for decoder,
      but added attention_weighted option using attention scores as weights.
"""

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
import importlib.util

# ============================================================
# CONFIG
# ============================================================
STRATEGY = "adversarial"  # "simple" | "contrastive" | "orthogonal" | "adversarial"

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUTPUT_DIR = f"lora_{STRATEGY}_output"
HF_TOKEN = os.getenv("HF_TOKEN", "")

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]  # reduced from 4 to 2 modules to save ~40% LoRA memory

EPOCHS = 4
BATCH_SIZE = 1            # reduced from 4 — contrastive does 2 forward passes per step
GRAD_ACCUM = 16           # effective batch = 1 * 16 = 16 (same as before)
LR = 2e-4
HEAD_LR = 5e-4
WARMUP_RATIO = 0.1
MAX_LEN = 384             # reduced from 512 — dataset max is 40 words (~60 tokens), 384 is plenty
SEED = 42

# Contrastive
LAMBDA_CONTRASTIVE = 0.5
CONTRASTIVE_LAYER_FRAC = 0.25
USE_NEGATIVE_PAIRS = True  # [FIX 3] add cross-validity negative pairs

# Orthogonal/Adversarial
LAMBDA_PLAUS = 0.3
LAMBDA_ORTH = 0.3          # [FIX 2] reduced from 1.0, gets normalized anyway
LAMBDA_DECORR = 0.1
LAMBDA_VAL_HEAD = 0.3
GRL_LAMBDA = 1.0
HEAD_PROJ_DIM = 256
EXTRACT_LAYER = -1

# [FIX 4] "last_token" | "mean_pool" | "attention_weighted"
POOL_MODE = "last_token"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def log_gpu_mem(tag=""):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"  [GPU {tag}] allocated={alloc:.2f}GB reserved={reserved:.2f}GB")

def load_json(p):
    with open(p, "r", encoding="utf-8") as f: return json.load(f)

def save_json(o, p):
    os.makedirs(os.path.dirname(p) or ".", exist_ok=True)
    with open(p, "w", encoding="utf-8") as f: json.dump(o, f, indent=2, ensure_ascii=False)


# ============================================================
# DATASETS
# ============================================================

class SimpleDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def _prompt(self, syl):
        msgs = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syl}\n\nAnswer yes or no."}
        ]
        return self.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def __getitem__(self, idx):
        ex = self.data[idx]
        prompt = self._prompt(ex["syllogism"])
        target = " yes" if ex["validity"] else " no"
        full = prompt + target
        enc = self.tokenizer(full, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
        penc = self.tokenizer(prompt, truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": enc.input_ids.squeeze(0),
            "attention_mask": enc.attention_mask.squeeze(0),
            "prompt_len": penc.input_ids.shape[1],
            "validity": 1 if ex["validity"] else 0,
            "plausibility": 1 if ex["plausibility"] else 0,
        }


# [FIX 3] Improved pairing with negative examples
class PairDataset(Dataset):
    def __init__(self, data, tokenizer, max_len, use_negatives=True):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.pairs = []

        vp = [x for x in data if x["validity"] and x["plausibility"]]
        vi = [x for x in data if x["validity"] and not x["plausibility"]]
        ip = [x for x in data if not x["validity"] and x["plausibility"]]
        ii = [x for x in data if not x["validity"] and not x["plausibility"]]

        rng = random.Random(SEED)

        # Positive pairs: same validity, different plausibility → push together
        def pos_pair(a, b, val):
            rng.shuffle(a); rng.shuffle(b)
            n = min(len(a), len(b))
            for i in range(n):
                self.pairs.append({
                    "syl_a": a[i]["syllogism"], "syl_b": b[i]["syllogism"],
                    "validity": val, "pair_type": "positive"
                })

        pos_pair(list(vp), list(vi), True)
        pos_pair(list(ip), list(ii), False)

        # Negative pairs: same plausibility, different validity → push apart
        if use_negatives:
            def neg_pair(a, b):
                rng.shuffle(a); rng.shuffle(b)
                n = min(len(a), len(b))
                for i in range(n):
                    self.pairs.append({
                        "syl_a": a[i]["syllogism"], "syl_b": b[i]["syllogism"],
                        "validity_a": a[i]["validity"], "validity_b": b[i]["validity"],
                        "pair_type": "negative"
                    })
            neg_pair(list(vp), list(ip))  # plausible: valid vs invalid
            neg_pair(list(vi), list(ii))  # implausible: valid vs invalid

        rng.shuffle(self.pairs)
        n_pos = sum(1 for p in self.pairs if p["pair_type"] == "positive")
        n_neg = sum(1 for p in self.pairs if p["pair_type"] == "negative")
        print(f"Pairs: {n_pos} positive (push together), {n_neg} negative (push apart)")

    def __len__(self): return len(self.pairs)

    def _prompt(self, syl):
        msgs = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syl}\n\nAnswer yes or no."}
        ]
        return self.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def __getitem__(self, idx):
        p = self.pairs[idx]
        out = {"pair_type": 0 if p["pair_type"] == "positive" else 1}

        if p["pair_type"] == "positive":
            target_a = " yes" if p["validity"] else " no"
            target_b = target_a
        else:
            target_a = " yes" if p["validity_a"] else " no"
            target_b = " yes" if p["validity_b"] else " no"

        for suffix, syl, target in [("_a", p["syl_a"], target_a), ("_b", p["syl_b"], target_b)]:
            prompt = self._prompt(syl)
            full = prompt + target
            enc = self.tokenizer(full, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
            penc = self.tokenizer(prompt, truncation=True, max_length=self.max_len, return_tensors="pt")
            out[f"input_ids{suffix}"] = enc.input_ids.squeeze(0)
            out[f"attention_mask{suffix}"] = enc.attention_mask.squeeze(0)
            out[f"prompt_len{suffix}"] = penc.input_ids.shape[1]

        return out


# ============================================================
# HEADS & GRADIENT REVERSAL
# ============================================================

class GRLFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.clone()
    @staticmethod
    def backward(ctx, grad):
        return -ctx.lam * grad, None

class GRL(nn.Module):
    def __init__(self, lam=1.0): super().__init__(); self.lam = lam
    def forward(self, x): return GRLFunction.apply(x, self.lam)

class DisentangleHeads(nn.Module):
    def __init__(self, hidden_dim, proj_dim, use_grl=False, grl_lam=1.0):
        super().__init__()
        self.val_proj = nn.Sequential(nn.Linear(hidden_dim, proj_dim), nn.ReLU(), nn.Dropout(0.1))
        self.val_cls = nn.Linear(proj_dim, 2)
        self.plaus_proj = nn.Sequential(nn.Linear(hidden_dim, proj_dim), nn.ReLU(), nn.Dropout(0.1))
        self.plaus_cls = nn.Linear(proj_dim, 2)
        self.grl = GRL(grl_lam) if use_grl else nn.Identity()

    def forward(self, h):
        hv = self.val_proj(h)
        hp = self.plaus_proj(self.grl(h))
        return self.val_cls(hv), self.plaus_cls(hp), hv, hp


# ============================================================
# LOSS HELPERS
# ============================================================

def causal_lm_loss(logits, input_ids, attention_mask, prompt_len):
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()
    mask = torch.zeros_like(shift_labels, dtype=torch.float32)
    for i in range(input_ids.shape[0]):
        pl = prompt_len[i].item() if isinstance(prompt_len, torch.Tensor) else prompt_len[i]
        end = attention_mask[i].sum().item() - 1
        mask[i, max(pl - 1, 0):end] = 1.0
    loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), reduction="none").view(shift_labels.shape)
    return (loss * mask).sum() / (mask.sum() + 1e-8)


# [FIX 3] Updated to handle positive (push together) and negative (push apart) pairs
def contrastive_repr_loss(hidden_a, hidden_b, mask_a, mask_b, layer_frac, pair_type):
    n_layers = len(hidden_a)
    start = int(n_layers * (1.0 - layer_frac))
    total = 0.0
    count = 0
    for li in range(start, n_layers):
        ha, hb = hidden_a[li], hidden_b[li]
        pos_a = mask_a.sum(dim=1) - 1
        pos_b = mask_b.sum(dim=1) - 1
        ra = ha[torch.arange(ha.size(0)), pos_a].float()
        rb = hb[torch.arange(hb.size(0)), pos_b].float()
        cos_sim = F.cosine_similarity(ra, rb, dim=-1)

        # Per-example: positive pairs → minimize distance, negative → maximize distance
        # pair_type: 0=positive, 1=negative
        pt = pair_type.float()
        # positive: loss = 1 - cos_sim  (push together)
        # negative: loss = max(0, cos_sim - margin)  (push apart)
        margin = 0.2
        pos_loss = (1.0 - cos_sim) * (1.0 - pt)
        neg_loss = torch.clamp(cos_sim - margin, min=0.0) * pt
        total += (pos_loss + neg_loss).mean()
        count += 1
    return total / max(count, 1)


def orth_loss(hv, hp):
    hv_n = F.normalize(hv, dim=-1)
    hp_n = F.normalize(hp, dim=-1)
    cc = (hv_n.T @ hp_n) / hv.shape[0]
    return torch.sum(cc ** 2)


def decorr_loss(hv, hp):
    def off_diag(h):
        hn = F.normalize(h, dim=0)
        c = hn.T @ hn / h.shape[0]
        m = ~torch.eye(c.shape[0], dtype=torch.bool, device=c.device)
        return torch.sum(c[m] ** 2)
    return off_diag(hv) + off_diag(hp)


# [FIX 4] Pooling with attention_weighted option
def pool_repr(hidden_states, attention_mask, layer_idx=-1, mode="last_token", attentions=None):
    h = hidden_states[layer_idx]  # (batch, seq, hidden)

    if mode == "last_token":
        pos = attention_mask.sum(dim=1) - 1
        return h[torch.arange(h.size(0), device=h.device), pos].float()

    elif mode == "mean_pool":
        mask = attention_mask.unsqueeze(-1).float()
        return ((h * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-8)).float()

    elif mode == "attention_weighted":
        # Use attention weights from last layer as importance scores
        # attentions[-1] shape: (batch, heads, seq, seq)
        if attentions is not None and len(attentions) > 0:
            attn = attentions[-1]  # last layer attention
            # Average across heads, take attention TO the last token
            last_pos = attention_mask.sum(dim=1) - 1  # (batch,)
            # Gather attention weights for last token attending to all positions
            attn_weights = torch.zeros(h.size(0), h.size(1), device=h.device)
            for i in range(h.size(0)):
                lp = last_pos[i].item()
                # Mean across heads: how much the last token attends to each position
                attn_weights[i, :lp+1] = attn[i, :, lp, :lp+1].mean(dim=0)
            # Normalize
            attn_weights = attn_weights * attention_mask.float()
            attn_weights = attn_weights / (attn_weights.sum(dim=1, keepdim=True) + 1e-8)
            # Weighted sum
            return (h * attn_weights.unsqueeze(-1)).sum(dim=1).float()
        else:
            # Fallback to last_token if no attention weights
            pos = attention_mask.sum(dim=1) - 1
            return h[torch.arange(h.size(0), device=h.device), pos].float()

    raise ValueError(f"Unknown pool mode: {mode}")


# [FIX 2] Loss normalizer: tracks running stats to keep losses on similar scales
class LossNormalizer:
    """
    Tracks EMA of each loss component's magnitude.
    Divides each loss by its EMA so all components are ~1.0 before weighting.
    First warmup_steps use simple averaging instead of EMA for stability.
    """
    def __init__(self, keys, momentum=0.9, warmup_steps=50):
        self.momentum = momentum
        self.warmup_steps = warmup_steps
        self.ema = {k: 1.0 for k in keys}
        self.initialized = {k: False for k in keys}
        self.step_count = {k: 0 for k in keys}
        self.warmup_sum = {k: 0.0 for k in keys}

    def normalize(self, key, loss_val):
        v = loss_val.detach().item()
        if v < 1e-8:
            return loss_val
        self.step_count[key] += 1
        if self.step_count[key] <= self.warmup_steps:
            self.warmup_sum[key] += v
            self.ema[key] = self.warmup_sum[key] / self.step_count[key]
            self.initialized[key] = True
        else:
            self.ema[key] = self.momentum * self.ema[key] + (1 - self.momentum) * v
            self.initialized[key] = True
        return loss_val / (self.ema[key] + 1e-8)


# ============================================================
# TRAINING
# ============================================================

def train():
    set_seed(SEED)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    token = HF_TOKEN or None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Strategy: {STRATEGY}")
    print(f"Model: {MODEL_NAME}")
    print(f"Pool mode: {POOL_MODE}")

    # [FIX 1] Single AutoModelForCausalLM for both generation loss and representations.
    # We use logits for causal LM loss and hidden_states (pre-lm_head) for
    # representation learning via the disentanglement heads.
    # No separate AutoModel needed — hidden_states are encoder representations.
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, token=token,
        torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    )
    model = model.to(DEVICE)
    lora_cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
                          lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGET_MODULES, bias="none")
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    # Enable gradient checkpointing: trades compute for ~40% less activation memory
    model.gradient_checkpointing_enable()
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    torch.cuda.empty_cache()
    log_gpu_mem("after model+lora")

    train_data = load_json(TRAIN_JSON)
    use_heads = STRATEGY in ("orthogonal", "adversarial")
    use_pairs = STRATEGY == "contrastive"
    need_attn = (POOL_MODE == "attention_weighted")

    if use_pairs:
        dataset = PairDataset(train_data, tokenizer, MAX_LEN, use_negatives=USE_NEGATIVE_PAIRS)
    else:
        dataset = SimpleDataset(train_data, tokenizer, MAX_LEN)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

    heads = None
    param_groups = [{"params": model.parameters(), "lr": LR, "weight_decay": 0.01}]
    if use_heads:
        heads = DisentangleHeads(
            model.config.hidden_size, HEAD_PROJ_DIM,
            use_grl=(STRATEGY == "adversarial"), grl_lam=GRL_LAMBDA
        ).to(DEVICE)
        param_groups.append({"params": heads.parameters(), "lr": HEAD_LR, "weight_decay": 0.01})
        print(f"Head params: {sum(p.numel() for p in heads.parameters()):,}")

    optimizer = torch.optim.AdamW(param_groups)
    total_steps = (len(dataloader) * EPOCHS) // GRAD_ACCUM
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)

    # [FIX 2] Initialize loss normalizer
    loss_keys = ["cls", "con", "vh", "ph", "ort", "dec"]
    loss_norm = LossNormalizer(loss_keys)

    print(f"Dataset: {len(dataset)} examples, {len(dataloader)} batches/epoch")
    print(f"Total steps: {total_steps}\n")

    model.train()
    if heads: heads.train()
    history = []

    for epoch in range(EPOCHS):
        m = {"cls": 0, "con": 0, "val_h": 0, "plaus_h": 0, "ort": 0, "dec": 0, "total": 0, "n": 0}

        for bi, batch in enumerate(dataloader):
            # ---- SIMPLE / ORTHOGONAL / ADVERSARIAL ----
            if not use_pairs:
                ids = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                plen = batch["prompt_len"]
                val_lbl = batch["validity"].to(DEVICE)
                plaus_lbl = batch["plausibility"].to(DEVICE)

                out = model(ids, attention_mask=mask,
                            output_hidden_states=use_heads,
                            output_attentions=need_attn,
                            use_cache=False)

                cls = causal_lm_loss(out.logits, ids, mask, plen)
                # [FIX 2] Normalize before weighting
                loss = loss_norm.normalize("cls", cls) * 1.0
                m["cls"] += cls.item() * ids.shape[0]

                if use_heads:
                    attns = out.attentions if need_attn else None
                    rep = pool_repr(out.hidden_states, mask, EXTRACT_LAYER, POOL_MODE, attns)
                    vl, pl, hv, hp = heads(rep)
                    vh_loss = F.cross_entropy(vl, val_lbl)
                    ph_loss = F.cross_entropy(pl, plaus_lbl)
                    ol = orth_loss(hv, hp)
                    dl = decorr_loss(hv, hp)

                    # [FIX 2] Each component normalized then weighted
                    loss = loss + LAMBDA_VAL_HEAD * loss_norm.normalize("vh", vh_loss)
                    loss = loss + LAMBDA_PLAUS * loss_norm.normalize("ph", ph_loss)
                    loss = loss + LAMBDA_ORTH * loss_norm.normalize("ort", ol)
                    loss = loss + LAMBDA_DECORR * loss_norm.normalize("dec", dl)

                    m["val_h"] += vh_loss.item() * ids.shape[0]
                    m["plaus_h"] += ph_loss.item() * ids.shape[0]
                    m["ort"] += ol.item() * ids.shape[0]
                    m["dec"] += dl.item() * ids.shape[0]

                m["total"] += loss.item() * ids.shape[0]
                m["n"] += ids.shape[0]

            # ---- CONTRASTIVE ----
            else:
                ids_a = batch["input_ids_a"].to(DEVICE)
                mask_a = batch["attention_mask_a"].to(DEVICE)
                plen_a = batch["prompt_len_a"]
                ids_b = batch["input_ids_b"].to(DEVICE)
                mask_b = batch["attention_mask_b"].to(DEVICE)
                plen_b = batch["prompt_len_b"]
                pair_type = batch["pair_type"].to(DEVICE)

                # Sequential forward passes to halve peak memory.
                # Pass A: compute CLS loss with grad, cache hidden states for contrastive loss
                out_a = model(ids_a, attention_mask=mask_a, output_hidden_states=True, use_cache=False)
                cls_a = causal_lm_loss(out_a.logits, ids_a, mask_a, plen_a)
                # Detach hidden states — no grad through them for contrastive loss
                hidden_a = tuple(h.detach() for h in out_a.hidden_states)
                del out_a
                torch.cuda.empty_cache()

                # Pass B: compute CLS loss with grad, cache hidden states
                out_b = model(ids_b, attention_mask=mask_b, output_hidden_states=True, use_cache=False)
                cls_b = causal_lm_loss(out_b.logits, ids_b, mask_b, plen_b)
                hidden_b = tuple(h.detach() for h in out_b.hidden_states)
                del out_b
                torch.cuda.empty_cache()

                cls = (cls_a + cls_b) / 2.0

                # Contrastive loss on detached hiddens (no second-order grad needed)
                con = contrastive_repr_loss(
                    hidden_a, hidden_b,
                    mask_a, mask_b, CONTRASTIVE_LAYER_FRAC, pair_type
                )
                del hidden_a, hidden_b

                # [FIX 2] Normalized loss mixing
                loss = loss_norm.normalize("cls", cls) * 1.0 + LAMBDA_CONTRASTIVE * loss_norm.normalize("con", con)

                bs = ids_a.shape[0]
                m["cls"] += cls.item() * bs
                m["con"] += con.item() * bs
                m["total"] += loss.item() * bs
                m["n"] += bs

            (loss / GRAD_ACCUM).backward()

            if (bi + 1) % GRAD_ACCUM == 0:
                params = list(model.parameters()) + (list(heads.parameters()) if heads else [])
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            if (bi + 1) % 30 == 0:
                n = m["n"]
                parts = [f"cls={m['cls']/n:.4f}"]
                if use_pairs: parts.append(f"con={m['con']/n:.4f}")
                if use_heads: parts.extend([f"vh={m['val_h']/n:.4f}", f"ph={m['plaus_h']/n:.4f}", f"ort={m['ort']/n:.6f}"])
                # [FIX 2] Show EMA scales so you can monitor normalization
                scales = " | scales: " + " ".join(f"{k}={loss_norm.ema[k]:.4f}" for k in loss_keys if loss_norm.initialized[k])
                print(f"  E{epoch+1} [{bi+1}/{len(dataloader)}] {' '.join(parts)}{scales}")

        n = m["n"]
        ep = {"epoch": epoch + 1, "cls": m["cls"]/n, "total": m["total"]/n}
        if use_pairs: ep["contrastive"] = m["con"]/n
        if use_heads: ep.update({"val_head": m["val_h"]/n, "plaus_head": m["plaus_h"]/n, "orth": m["ort"]/n})
        history.append(ep)
        print(f"\n  Epoch {epoch+1}: {ep}\n")

        ckpt = os.path.join(OUTPUT_DIR, f"epoch{epoch+1}")
        model.save_pretrained(ckpt); tokenizer.save_pretrained(ckpt)
        if heads: torch.save(heads.state_dict(), os.path.join(ckpt, "heads.pt"))

    final = os.path.join(OUTPUT_DIR, "final")
    model.save_pretrained(final); tokenizer.save_pretrained(final)
    if heads: torch.save(heads.state_dict(), os.path.join(final, "heads.pt"))
    save_json(history, os.path.join(OUTPUT_DIR, "history.json"))
    print(f"Saved to {final}")
    return model, tokenizer, heads


# ============================================================
# INFERENCE
# ============================================================

def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        logits = model(full_ids, use_cache=False).logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    return lp[0, max(prompt_ids.shape[1] - 1, 0):].sum().item()


@torch.no_grad()
def evaluate(model, tokenizer, test_path, out_path):
    model.eval()
    data = load_json(test_path)
    preds = []
    for i, ex in enumerate(data):
        msgs = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."}
        ]
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        y = score_label(model, tokenizer, prompt, " yes", MAX_LEN)
        n = score_label(model, tokenizer, prompt, " no", MAX_LEN)
        preds.append({"id": ex["id"], "validity": bool(y >= n)})
        if (i + 1) % 50 == 0: print(f"  {i+1}/{len(data)}")
    save_json(preds, out_path)
    print(f"Predictions: {out_path}")
    return preds


def run_eval(pred_path):
    if not os.path.exists(EVAL_SCRIPT):
        print("Eval script not found, skipping official eval")
        return None
    spec = importlib.util.spec_from_file_location("ev", EVAL_SCRIPT)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    out = os.path.join(OUTPUT_DIR, "eval_results.json")
    mod.run_full_scoring(TEST_JSON, pred_path, out)
    r = load_json(out)
    print(f"ACC={r['accuracy']:.2f}  TCE={r['content_effect']:.4f}  Score={r['combined_score']:.4f}")
    return r


def subgroup_acc(test_path, pred_path):
    data = load_json(test_path)
    preds = {p["id"]: p["validity"] for p in load_json(pred_path)}
    buckets = {}
    for ex in data:
        v = "valid" if ex["validity"] else "invalid"
        p = "plausible" if ex["plausibility"] else "implausible"
        k = f"{p}_{v}"
        buckets.setdefault(k, {"c": 0, "t": 0})
        buckets[k]["t"] += 1
        if preds.get(ex["id"]) == ex["validity"]:
            buckets[k]["c"] += 1
    print("\nSubgroup accuracy:")
    for k in sorted(buckets):
        b = buckets[k]
        print(f"  {k:30s}: {100*b['c']/b['t']:6.2f}% ({b['c']}/{b['t']})")


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    model, tokenizer, heads = train()
    pred_path = os.path.join(OUTPUT_DIR, "predictions.json")
    evaluate(model, tokenizer, TEST_JSON, pred_path)
    subgroup_acc(TEST_JSON, pred_path)
    run_eval(pred_path)

Strategy: adversarial
Model: Qwen/Qwen2.5-3B-Instruct
Pool mode: last_token


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 3,686,400 || all params: 3,089,625,088 || trainable%: 0.1193
  [GPU after model+lora] allocated=11.65GB reserved=12.29GB
Head params: 1,050,116
Dataset: 960 examples, 960 batches/epoch
Total steps: 240

  E1 [30/960] cls=18.2417 vh=0.8406 ph=0.7090 ort=1.000000 | scales: cls=18.2417 vh=0.8406 ph=0.7090 ort=1.0000 dec=29487.6000
  E1 [60/960] cls=18.0375 vh=0.7887 ph=0.6734 ort=1.000000 | scales: cls=17.6018 vh=0.8269 ph=0.6803 ort=1.0000 dec=28814.6877
  E1 [90/960] cls=18.8410 vh=0.8625 ph=0.7223 ort=1.000000 | scales: cls=20.5902 vh=1.0710 ph=0.8190 ort=1.0000 dec=27424.7030
  E1 [120/960] cls=18.3891 vh=0.8486 ph=0.7240 ort=1.000000 | scales: cls=16.9898 vh=0.7201 ph=0.7909 ort=1.0000 dec=26556.2009
  E1 [150/960] cls=18.0579 vh=0.7825 ph=0.7251 ort=1.000000 | scales: cls=17.0015 vh=0.4998 ph=0.7487 ort=1.0000 dec=26325.7902
  E1 [180/960] cls=17.6774 vh=0.7464 ph=0.7292 ort=1.000000 | scales: cls=15.4355 vh=0.3968 ph=0.7388 ort=1.0000 dec=25952.6567
  E1 [210/960]

In [10]:
import os
import json
from glob import glob

ROOT = "."  # change if needed, e.g. "/content/SemEval_dataset"

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def count_examples(obj):
    if isinstance(obj, list):
        return len(obj)
    if isinstance(obj, dict):
        # common fallback: if json is a dict with one top-level list
        for v in obj.values():
            if isinstance(v, list):
                return len(v)
    raise ValueError(f"Unsupported JSON format in file")

train_files = sorted(glob(os.path.join(ROOT, "train_data", "**", "*.json"), recursive=True))
test_files  = sorted(glob(os.path.join(ROOT, "test_data",  "**", "*.json"), recursive=True))

train_total = 0
test_total = 0

print("TRAIN FILES")
for fp in train_files:
    n = count_examples(load_json(fp))
    train_total += n
    print(f"{fp}: {n}")

print("\nTEST FILES")
for fp in test_files:
    n = count_examples(load_json(fp))
    test_total += n
    print(f"{fp}: {n}")

print("\nTOTALS")
print(f"Train total syllogisms: {train_total}")
print(f"Test total syllogisms:  {test_total}")
print(f"Grand total:            {train_total + test_total}")

TRAIN FILES
./train_data/subtask 1/train_data.json: 960

TEST FILES
./test_data/subtask 1/test_data_subtask_1.json: 191
./test_data/subtask 2/test_data_subtask_2.json: 190
./test_data/subtask 3/test_data_subtask_3.json: 192
./test_data/subtask 4/test_data_subtask_4.json: 192

TOTALS
Train total syllogisms: 960
Test total syllogisms:  765
Grand total:            1725


In [15]:
"""
Verify token positions during extraction vs inference.
+ Compare last vs second-last prompt token representations.
"""

import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
HF_TOKEN = os.getenv("HF_TOKEN", "")
MAX_LEN = 512
YES_TEXT = " yes"
NO_TEXT = " no"

SAMPLE_SYLLOGISM = "All cars are a type of vehicle. No animal is a car. Therefore, no animal can be a vehicle."


def build_prompt(tokenizer, syllogism):
    messages = [
        {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
        {"role": "user", "content": f"Argument:\n{syllogism}\n\nAnswer yes or no."}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def main():
    token = HF_TOKEN or None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    prompt = build_prompt(tokenizer, SAMPLE_SYLLOGISM)

    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False,
                           truncation=True, max_length=MAX_LEN).input_ids
    full_yes_ids = tokenizer(prompt + YES_TEXT, return_tensors="pt", add_special_tokens=False,
                             truncation=True, max_length=MAX_LEN).input_ids
    full_no_ids = tokenizer(prompt + NO_TEXT, return_tensors="pt", add_special_tokens=False,
                            truncation=True, max_length=MAX_LEN).input_ids

    prompt_len = prompt_ids.shape[1]
    yes_len = full_yes_ids.shape[1]
    no_len = full_no_ids.shape[1]

    yes_label_tokens = yes_len - prompt_len
    no_label_tokens = no_len - prompt_len

    print("=" * 60)
    print("  TOKEN POSITION VERIFICATION")
    print("=" * 60)

    print(f"\nPrompt length: {prompt_len} tokens")
    print(f"Prompt + ' yes': {yes_len} tokens ({yes_label_tokens} label tokens)")
    print(f"Prompt + ' no':  {no_len} tokens ({no_label_tokens} label tokens)")

    print(f"\n--- Last 5 prompt tokens ---")
    for i in range(max(0, prompt_len - 5), prompt_len):
        tid = prompt_ids[0, i].item()
        print(f"  pos {i}: id={tid} -> '{tokenizer.decode([tid])}'")

    print(f"\n--- Last 5 tokens of prompt+yes ---")
    for i in range(max(0, yes_len - 5), yes_len):
        tid = full_yes_ids[0, i].item()
        marker = " <-- LABEL" if i >= prompt_len else ""
        print(f"  pos {i}: id={tid} -> '{tokenizer.decode([tid])}'{marker}")

    print(f"\n--- Last 5 tokens of prompt+no ---")
    for i in range(max(0, no_len - 5), no_len):
        tid = full_no_ids[0, i].item()
        marker = " <-- LABEL" if i >= prompt_len else ""
        print(f"  pos {i}: id={tid} -> '{tokenizer.decode([tid])}'{marker}")

    print(f"\n{'=' * 60}")
    print(f"  EXTRACTION")
    print(f"{'=' * 60}")
    print(f"  Last prompt token position: {prompt_len - 1}")

    print(f"\n{'=' * 60}")
    print(f"  LOAD MODEL")
    print(f"{'=' * 60}")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=token,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )
    model.eval()
    model.config.use_cache = False

    device = model.device

    prompt_enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)
    full_enc = tokenizer(prompt + YES_TEXT, return_tensors="pt", add_special_tokens=False).to(device)

    with torch.no_grad():
        out_prompt = model(**prompt_enc, output_hidden_states=True, use_cache=False)
        out_full = model(**full_enc, output_hidden_states=True, use_cache=False)

    last_layer = len(out_prompt.hidden_states) - 1

    print(f"\n{'=' * 60}")
    print(f"  EXTRACTION vs INFERENCE CHECK")
    print(f"{'=' * 60}")

    h_extract = out_prompt.hidden_states[last_layer][0, -1, :].float()
    h_infer_last = out_full.hidden_states[last_layer][0, -1, :].float()
    h_infer_prompt_end = out_full.hidden_states[last_layer][0, prompt_len - 1, :].float()

    cos1 = torch.nn.functional.cosine_similarity(
        h_extract.unsqueeze(0), h_infer_last.unsqueeze(0)).item()

    cos2 = torch.nn.functional.cosine_similarity(
        h_extract.unsqueeze(0), h_infer_prompt_end.unsqueeze(0)).item()

    print(f"Cosine (extract vs answer token):     {cos1:.6f}")
    print(f"Cosine (extract vs prompt end):       {cos2:.6f}")

    print(f"\n{'=' * 60}")
    print(f"  LAST vs SECOND-LAST TOKEN")
    print(f"{'=' * 60}")

    h_last = out_prompt.hidden_states[last_layer][0, -1, :].float()
    h_second_last = out_prompt.hidden_states[last_layer][0, -2, :].float()

    last_token_id = prompt_ids[0, -1].item()
    second_last_token_id = prompt_ids[0, -2].item()

    print(f"\nLast token (-1):        '{tokenizer.decode([last_token_id])}'")
    print(f"Second-last token (-2): '{tokenizer.decode([second_last_token_id])}'")

    cos_sim = torch.nn.functional.cosine_similarity(
        h_last.unsqueeze(0), h_second_last.unsqueeze(0)
    ).item()

    l2_dist = (h_last - h_second_last).norm().item()

    print(f"\nCosine similarity: {cos_sim:.6f}")
    print(f"L2 distance:       {l2_dist:.4f}")

    print(f"\nLayer-wise cosine similarity:")
    for layer in range(len(out_prompt.hidden_states)):
        h_l = out_prompt.hidden_states[layer][0, -1, :].float()
        h_sl = out_prompt.hidden_states[layer][0, -2, :].float()

        cos = torch.nn.functional.cosine_similarity(
            h_l.unsqueeze(0), h_sl.unsqueeze(0)
        ).item()

        print(f"Layer {layer:02d}: {cos:.6f}")


if __name__ == "__main__":
    main()

  TOKEN POSITION VERIFICATION

Prompt length: 78 tokens
Prompt + ' yes': 79 tokens (1 label tokens)
Prompt + ' no':  79 tokens (1 label tokens)

--- Last 5 prompt tokens ---
  pos 73: id=151645 -> '<|im_end|>'
  pos 74: id=198 -> '
'
  pos 75: id=151644 -> '<|im_start|>'
  pos 76: id=77091 -> 'assistant'
  pos 77: id=198 -> '
'

--- Last 5 tokens of prompt+yes ---
  pos 74: id=198 -> '
'
  pos 75: id=151644 -> '<|im_start|>'
  pos 76: id=77091 -> 'assistant'
  pos 77: id=198 -> '
'
  pos 78: id=9834 -> ' yes' <-- LABEL

--- Last 5 tokens of prompt+no ---
  pos 74: id=198 -> '
'
  pos 75: id=151644 -> '<|im_start|>'
  pos 76: id=77091 -> 'assistant'
  pos 77: id=198 -> '
'
  pos 78: id=902 -> ' no' <-- LABEL

  EXTRACTION
  Last prompt token position: 77

  LOAD MODEL


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]


  EXTRACTION vs INFERENCE CHECK
Cosine (extract vs answer token):     0.070081
Cosine (extract vs prompt end):       1.000000

  LAST vs SECOND-LAST TOKEN

Last token (-1):        '
'
Second-last token (-2): 'assistant'

Cosine similarity: 0.048328
L2 distance:       200.4432

Layer-wise cosine similarity:
Layer 00: -0.191200
Layer 01: 0.130626
Layer 02: 0.376099
Layer 03: 0.389327
Layer 04: 0.385623
Layer 05: 0.357974
Layer 06: 0.349965
Layer 07: 0.371214
Layer 08: 0.235692
Layer 09: 0.280679
Layer 10: 0.290138
Layer 11: 0.295846
Layer 12: 0.330980
Layer 13: 0.314945
Layer 14: 0.348455
Layer 15: 0.357063
Layer 16: 0.365093
Layer 17: 0.358720
Layer 18: 0.432116
Layer 19: 0.484179
Layer 20: 0.452990
Layer 21: 0.602802
Layer 22: 0.731407
Layer 23: 0.781187
Layer 24: 0.739900
Layer 25: 0.713808
Layer 26: 0.693171
Layer 27: 0.661288
Layer 28: 0.702869
Layer 29: 0.697029
Layer 30: 0.654439
Layer 31: 0.496560
Layer 32: 0.552065
Layer 33: 0.583971
Layer 34: 0.640580
Layer 35: 0.595465
Layer 

In [ ]:
print(f"\n{'=' * 60}")
print(f"  LAST vs SECOND-LAST TOKEN COMPARISON")
print(f"{'=' * 60}")

# Last layer index
last_layer = len(out_prompt.hidden_states) - 1

# Extract hidden states (prompt-only)
h_last = out_prompt.hidden_states[last_layer][0, -1, :].float()
h_second_last = out_prompt.hidden_states[last_layer][0, -2, :].float()

# Token IDs for reference
last_token_id = prompt_ids[0, -1].item()
second_last_token_id = prompt_ids[0, -2].item()

print(f"\n  Token positions:")
print(f"    Last token (-1): pos {prompt_len - 1} -> '{tokenizer.decode([last_token_id])}'")
print(f"    Second-last (-2): pos {prompt_len - 2} -> '{tokenizer.decode([second_last_token_id])}'")

# Cosine similarity
cos_sim = torch.nn.functional.cosine_similarity(
    h_last.unsqueeze(0), h_second_last.unsqueeze(0)
).item()

# L2 distance
l2_dist = (h_last - h_second_last).norm().item()

print(f"\n  Cosine similarity (last vs second-last): {cos_sim:.6f}")
print(f"  L2 distance (last vs second-last):      {l2_dist:.4f}")

# Optional: compare across multiple layers
print(f"\n  Layer-wise cosine similarity (last vs second-last):")
for layer in range(len(out_prompt.hidden_states)):
    h_l = out_prompt.hidden_states[layer][0, -1, :].float()
    h_sl = out_prompt.hidden_states[layer][0, -2, :].float()
    cos = torch.nn.functional.cosine_similarity(
        h_l.unsqueeze(0), h_sl.unsqueeze(0)
    ).item()
    print(f"    Layer {layer:02d}: {cos:.6f}")

In [1]:
"""
Unified LoRA Fine-Tuning for Syllogistic Reasoning
=====================================================
4 strategies selectable via STRATEGY flag:

  "simple"        = Option 1: validity classification only
  "contrastive"   = Option 2: validity + plausibility-invariance pairs
  "orthogonal"    = Option 3: multi-task with orthogonal separation
  "adversarial"   = Option 4: multi-task with gradient reversal

Fixes applied:
  [1] Single AutoModelForCausalLM for both generation loss and representations.
      We use logits for causal LM loss and hidden_states (pre-lm_head) for
      representation learning. No separate AutoModel needed.
  [2] Loss normalization: each loss component is scale-normalized before
      weighting so no single term dominates.
  [3] Improved contrastive pairing: cross-validity pairs added as negative
      examples (same plausibility, different validity) for harder contrast.
  [4] Weighted attention pooling: last_token still default for decoder,
      but added attention_weighted option using attention scores as weights.
"""

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import json
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType
import importlib.util

# ============================================================
# CONFIG
# ============================================================
STRATEGY = "contrastive"  # "simple" | "contrastive" | "orthogonal" | "adversarial"

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
TRAIN_JSON = "train_data/subtask 1/train_data.json"
TEST_JSON = "test_data/subtask 1/test_data_subtask_1.json"
EVAL_SCRIPT = "evaluation_kit/task 1 & 3/evaluation_script.py"
OUTPUT_DIR = f"lora_{STRATEGY}_output"
HF_TOKEN = os.getenv("HF_TOKEN", "")

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "v_proj"]  # reduced from 4 to 2 modules to save ~40% LoRA memory

EPOCHS = 5
BATCH_SIZE = 1            # reduced from 4 — contrastive does 2 forward passes per step
GRAD_ACCUM = 16           # effective batch = 1 * 16 = 16 (same as before)
LR = 2e-4
HEAD_LR = 5e-4
WARMUP_RATIO = 0.1
MAX_LEN = 384             # reduced from 512 — dataset max is 40 words (~60 tokens), 384 is plenty
SEED = 42

# Contrastive
LAMBDA_CONTRASTIVE = 0.5
CONTRASTIVE_LAYER_FRAC = 0.25
USE_NEGATIVE_PAIRS = True  # [FIX 3] add cross-validity negative pairs

# Orthogonal/Adversarial
LAMBDA_PLAUS = 0.3
LAMBDA_ORTH = 0.3          # [FIX 2] reduced from 1.0, gets normalized anyway
LAMBDA_DECORR = 0.1
LAMBDA_VAL_HEAD = 0.3
GRL_LAMBDA = 1.0
HEAD_PROJ_DIM = 256
EXTRACT_LAYER = -1

# [FIX 4] "last_token" | "mean_pool" | "attention_weighted"
POOL_MODE = "last_token"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def log_gpu_mem(tag=""):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"  [GPU {tag}] allocated={alloc:.2f}GB reserved={reserved:.2f}GB")

def load_json(p):
    with open(p, "r", encoding="utf-8") as f: return json.load(f)

def save_json(o, p):
    os.makedirs(os.path.dirname(p) or ".", exist_ok=True)
    with open(p, "w", encoding="utf-8") as f: json.dump(o, f, indent=2, ensure_ascii=False)


# ============================================================
# DATASETS
# ============================================================

class SimpleDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.data)

    def _prompt(self, syl):
        msgs = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syl}\n\nAnswer yes or no."}
        ]
        return self.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def __getitem__(self, idx):
        ex = self.data[idx]
        prompt = self._prompt(ex["syllogism"])
        target = " yes" if ex["validity"] else " no"
        full = prompt + target
        enc = self.tokenizer(full, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
        penc = self.tokenizer(prompt, truncation=True, max_length=self.max_len, return_tensors="pt")
        return {
            "input_ids": enc.input_ids.squeeze(0),
            "attention_mask": enc.attention_mask.squeeze(0),
            "prompt_len": penc.input_ids.shape[1],
            "validity": 1 if ex["validity"] else 0,
            "plausibility": 1 if ex["plausibility"] else 0,
        }


# [FIX 3] Improved pairing with negative examples
class PairDataset(Dataset):
    def __init__(self, data, tokenizer, max_len, use_negatives=True):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.pairs = []

        vp = [x for x in data if x["validity"] and x["plausibility"]]
        vi = [x for x in data if x["validity"] and not x["plausibility"]]
        ip = [x for x in data if not x["validity"] and x["plausibility"]]
        ii = [x for x in data if not x["validity"] and not x["plausibility"]]

        rng = random.Random(SEED)

        # Positive pairs: same validity, different plausibility → push together
        def pos_pair(a, b, val):
            rng.shuffle(a); rng.shuffle(b)
            n = min(len(a), len(b))
            for i in range(n):
                self.pairs.append({
                    "syl_a": a[i]["syllogism"], "syl_b": b[i]["syllogism"],
                    "validity": val, "pair_type": "positive"
                })

        pos_pair(list(vp), list(vi), True)
        pos_pair(list(ip), list(ii), False)

        # Negative pairs: same plausibility, different validity → push apart
        if use_negatives:
            def neg_pair(a, b):
                rng.shuffle(a); rng.shuffle(b)
                n = min(len(a), len(b))
                for i in range(n):
                    self.pairs.append({
                        "syl_a": a[i]["syllogism"], "syl_b": b[i]["syllogism"],
                        "validity_a": a[i]["validity"], "validity_b": b[i]["validity"],
                        "pair_type": "negative"
                    })
            neg_pair(list(vp), list(ip))  # plausible: valid vs invalid
            neg_pair(list(vi), list(ii))  # implausible: valid vs invalid

        rng.shuffle(self.pairs)
        n_pos = sum(1 for p in self.pairs if p["pair_type"] == "positive")
        n_neg = sum(1 for p in self.pairs if p["pair_type"] == "negative")
        print(f"Pairs: {n_pos} positive (push together), {n_neg} negative (push apart)")

    def __len__(self): return len(self.pairs)

    def _prompt(self, syl):
        msgs = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{syl}\n\nAnswer yes or no."}
        ]
        return self.tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

    def __getitem__(self, idx):
        p = self.pairs[idx]
        out = {"pair_type": 0 if p["pair_type"] == "positive" else 1}

        if p["pair_type"] == "positive":
            target_a = " yes" if p["validity"] else " no"
            target_b = target_a
        else:
            target_a = " yes" if p["validity_a"] else " no"
            target_b = " yes" if p["validity_b"] else " no"

        for suffix, syl, target in [("_a", p["syl_a"], target_a), ("_b", p["syl_b"], target_b)]:
            prompt = self._prompt(syl)
            full = prompt + target
            enc = self.tokenizer(full, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
            penc = self.tokenizer(prompt, truncation=True, max_length=self.max_len, return_tensors="pt")
            out[f"input_ids{suffix}"] = enc.input_ids.squeeze(0)
            out[f"attention_mask{suffix}"] = enc.attention_mask.squeeze(0)
            out[f"prompt_len{suffix}"] = penc.input_ids.shape[1]

        return out


# ============================================================
# HEADS & GRADIENT REVERSAL
# ============================================================

class GRLFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lam):
        ctx.lam = lam
        return x.clone()
    @staticmethod
    def backward(ctx, grad):
        return -ctx.lam * grad, None

class GRL(nn.Module):
    def __init__(self, lam=1.0): super().__init__(); self.lam = lam
    def forward(self, x): return GRLFunction.apply(x, self.lam)

class DisentangleHeads(nn.Module):
    def __init__(self, hidden_dim, proj_dim, use_grl=False, grl_lam=1.0):
        super().__init__()
        self.val_proj = nn.Sequential(nn.Linear(hidden_dim, proj_dim), nn.ReLU(), nn.Dropout(0.1))
        self.val_cls = nn.Linear(proj_dim, 2)
        self.plaus_proj = nn.Sequential(nn.Linear(hidden_dim, proj_dim), nn.ReLU(), nn.Dropout(0.1))
        self.plaus_cls = nn.Linear(proj_dim, 2)
        self.grl = GRL(grl_lam) if use_grl else nn.Identity()

    def forward(self, h):
        hv = self.val_proj(h)
        hp = self.plaus_proj(self.grl(h))
        return self.val_cls(hv), self.plaus_cls(hp), hv, hp


# ============================================================
# LOSS HELPERS
# ============================================================

def causal_lm_loss(logits, input_ids, attention_mask, prompt_len):
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = input_ids[:, 1:].contiguous()
    mask = torch.zeros_like(shift_labels, dtype=torch.float32)
    for i in range(input_ids.shape[0]):
        pl = prompt_len[i].item() if isinstance(prompt_len, torch.Tensor) else prompt_len[i]
        end = attention_mask[i].sum().item() - 1
        mask[i, max(pl - 1, 0):end] = 1.0
    loss = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), reduction="none").view(shift_labels.shape)
    return (loss * mask).sum() / (mask.sum() + 1e-8)


# [FIX 3] Updated to handle positive (push together) and negative (push apart) pairs
def contrastive_repr_loss(hidden_a, hidden_b, mask_a, mask_b, layer_frac, pair_type):
    n_layers = len(hidden_a)
    start = int(n_layers * (1.0 - layer_frac))
    total = 0.0
    count = 0
    for li in range(start, n_layers):
        ha, hb = hidden_a[li], hidden_b[li]
        pos_a = mask_a.sum(dim=1) - 1
        pos_b = mask_b.sum(dim=1) - 1
        ra = ha[torch.arange(ha.size(0)), pos_a].float()
        rb = hb[torch.arange(hb.size(0)), pos_b].float()
        cos_sim = F.cosine_similarity(ra, rb, dim=-1)

        # Per-example: positive pairs → minimize distance, negative → maximize distance
        # pair_type: 0=positive, 1=negative
        pt = pair_type.float()
        # positive: loss = 1 - cos_sim  (push together)
        # negative: loss = max(0, cos_sim - margin)  (push apart)
        margin = 0.2
        pos_loss = (1.0 - cos_sim) * (1.0 - pt)
        neg_loss = torch.clamp(cos_sim - margin, min=0.0) * pt
        total += (pos_loss + neg_loss).mean()
        count += 1
    return total / max(count, 1)


def orth_loss(hv, hp):
    hv_n = F.normalize(hv, dim=-1)
    hp_n = F.normalize(hp, dim=-1)
    cc = (hv_n.T @ hp_n) / hv.shape[0]
    return torch.sum(cc ** 2)


def decorr_loss(hv, hp):
    def off_diag(h):
        hn = F.normalize(h, dim=0)
        c = hn.T @ hn / h.shape[0]
        m = ~torch.eye(c.shape[0], dtype=torch.bool, device=c.device)
        return torch.sum(c[m] ** 2)
    return off_diag(hv) + off_diag(hp)


# [FIX 4] Pooling with attention_weighted option
def pool_repr(hidden_states, attention_mask, layer_idx=-1, mode="last_token", attentions=None):
    h = hidden_states[layer_idx]  # (batch, seq, hidden)

    if mode == "last_token":
        pos = attention_mask.sum(dim=1) - 1
        return h[torch.arange(h.size(0), device=h.device), pos].float()

    elif mode == "mean_pool":
        mask = attention_mask.unsqueeze(-1).float()
        return ((h * mask).sum(dim=1) / (mask.sum(dim=1) + 1e-8)).float()

    elif mode == "attention_weighted":
        # Use attention weights from last layer as importance scores
        # attentions[-1] shape: (batch, heads, seq, seq)
        if attentions is not None and len(attentions) > 0:
            attn = attentions[-1]  # last layer attention
            # Average across heads, take attention TO the last token
            last_pos = attention_mask.sum(dim=1) - 1  # (batch,)
            # Gather attention weights for last token attending to all positions
            attn_weights = torch.zeros(h.size(0), h.size(1), device=h.device)
            for i in range(h.size(0)):
                lp = last_pos[i].item()
                # Mean across heads: how much the last token attends to each position
                attn_weights[i, :lp+1] = attn[i, :, lp, :lp+1].mean(dim=0)
            # Normalize
            attn_weights = attn_weights * attention_mask.float()
            attn_weights = attn_weights / (attn_weights.sum(dim=1, keepdim=True) + 1e-8)
            # Weighted sum
            return (h * attn_weights.unsqueeze(-1)).sum(dim=1).float()
        else:
            # Fallback to last_token if no attention weights
            pos = attention_mask.sum(dim=1) - 1
            return h[torch.arange(h.size(0), device=h.device), pos].float()

    raise ValueError(f"Unknown pool mode: {mode}")


# [FIX 2] Loss normalizer: tracks running stats to keep losses on similar scales
class LossNormalizer:
    """
    Tracks EMA of each loss component's magnitude.
    Divides each loss by its EMA so all components are ~1.0 before weighting.
    First warmup_steps use simple averaging instead of EMA for stability.
    """
    def __init__(self, keys, momentum=0.9, warmup_steps=50):
        self.momentum = momentum
        self.warmup_steps = warmup_steps
        self.ema = {k: 1.0 for k in keys}
        self.initialized = {k: False for k in keys}
        self.step_count = {k: 0 for k in keys}
        self.warmup_sum = {k: 0.0 for k in keys}

    def normalize(self, key, loss_val):
        v = loss_val.detach().item()
        if v < 1e-8:
            return loss_val
        self.step_count[key] += 1
        if self.step_count[key] <= self.warmup_steps:
            self.warmup_sum[key] += v
            self.ema[key] = self.warmup_sum[key] / self.step_count[key]
            self.initialized[key] = True
        else:
            self.ema[key] = self.momentum * self.ema[key] + (1 - self.momentum) * v
            self.initialized[key] = True
        return loss_val / (self.ema[key] + 1e-8)


# ============================================================
# INFERENCE & EVALUATION HELPERS
# ============================================================

def score_label(model, tokenizer, prompt, label_text, max_len):
    full = prompt + label_text
    full_ids = tokenizer(full, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False, truncation=True, max_length=max_len).input_ids.to(model.device)
    with torch.no_grad():
        logits = model(full_ids, use_cache=False).logits[:, :-1, :]
        target = full_ids[:, 1:]
        lp = F.log_softmax(logits, dim=-1).gather(-1, target.unsqueeze(-1)).squeeze(-1)
    return lp[0, max(prompt_ids.shape[1] - 1, 0):].sum().item()


@torch.no_grad()
def evaluate(model, tokenizer, test_path, out_path):
    model.eval()
    data = load_json(test_path)
    preds = []
    for i, ex in enumerate(data):
        msgs = [
            {"role": "system", "content": "You are a strict formal logic reasoner. Decide only whether the conclusion logically follows from the premises. Ignore plausibility and world knowledge. Reply with only yes or no."},
            {"role": "user", "content": f"Argument:\n{ex['syllogism']}\n\nAnswer yes or no."}
        ]
        prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        y = score_label(model, tokenizer, prompt, " yes", MAX_LEN)
        n = score_label(model, tokenizer, prompt, " no", MAX_LEN)
        preds.append({"id": ex["id"], "validity": bool(y >= n)})
        if (i + 1) % 50 == 0: print(f"  {i+1}/{len(data)}")
    save_json(preds, out_path)
    print(f"Predictions: {out_path}")
    return preds


def run_eval(pred_path):
    if not os.path.exists(EVAL_SCRIPT):
        print("Eval script not found, skipping official eval")
        return None
    spec = importlib.util.spec_from_file_location("ev", EVAL_SCRIPT)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    out = os.path.join(OUTPUT_DIR, "eval_results.json")
    mod.run_full_scoring(TEST_JSON, pred_path, out)
    r = load_json(out)
    print(f"ACC={r['accuracy']:.2f}  TCE={r['content_effect']:.4f}  Score={r['combined_score']:.4f}")
    return r


def subgroup_acc(test_path, pred_path):
    data = load_json(test_path)
    preds = {p["id"]: p["validity"] for p in load_json(pred_path)}
    buckets = {}
    for ex in data:
        v = "valid" if ex["validity"] else "invalid"
        p = "plausible" if ex["plausibility"] else "implausible"
        k = f"{p}_{v}"
        buckets.setdefault(k, {"c": 0, "t": 0})
        buckets[k]["t"] += 1
        if preds.get(ex["id"]) == ex["validity"]:
            buckets[k]["c"] += 1
    print("\nSubgroup accuracy:")
    for k in sorted(buckets):
        b = buckets[k]
        print(f"  {k:30s}: {100*b['c']/b['t']:6.2f}% ({b['c']}/{b['t']})")


def run_checkpoint_eval(model, tokenizer, label, history_entry):
    """
    Run full evaluation (ACC, CE, Score) at a checkpoint.
    Switches model to eval, runs inference, runs official eval, then restores train mode.
    Returns the eval results dict (or None if eval script missing).
    """
    print(f"\n{'='*60}")
    print(f"  EVALUATION: {label}")
    print(f"{'='*60}")

    pred_path = os.path.join(OUTPUT_DIR, f"predictions_{label}.json")
    evaluate(model, tokenizer, TEST_JSON, pred_path)
    subgroup_acc(TEST_JSON, pred_path)
    result = run_eval(pred_path)

    # Store metrics in history entry
    if result is not None:
        history_entry["accuracy"] = result["accuracy"]
        history_entry["content_effect"] = result["content_effect"]
        history_entry["combined_score"] = result["combined_score"]

    # Restore train mode
    model.train()
    print(f"{'='*60}\n")
    return result


# ============================================================
# TRAINING
# ============================================================

def train():
    set_seed(SEED)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    token = HF_TOKEN or None
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=token)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Strategy: {STRATEGY}")
    print(f"Model: {MODEL_NAME}")
    print(f"Pool mode: {POOL_MODE}")

    # [FIX 1] Single AutoModelForCausalLM for both generation loss and representations.
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, token=token,
        torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,
    )
    model = model.to(DEVICE)
    lora_cfg = LoraConfig(task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
                          lora_dropout=LORA_DROPOUT, target_modules=LORA_TARGET_MODULES, bias="none")
    model = get_peft_model(model, lora_cfg)
    model.print_trainable_parameters()

    # Enable gradient checkpointing: trades compute for ~40% less activation memory
    model.gradient_checkpointing_enable()
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    torch.cuda.empty_cache()
    log_gpu_mem("after model+lora")

    train_data = load_json(TRAIN_JSON)
    use_heads = STRATEGY in ("orthogonal", "adversarial")
    use_pairs = STRATEGY == "contrastive"
    need_attn = (POOL_MODE == "attention_weighted")

    if use_pairs:
        dataset = PairDataset(train_data, tokenizer, MAX_LEN, use_negatives=USE_NEGATIVE_PAIRS)
    else:
        dataset = SimpleDataset(train_data, tokenizer, MAX_LEN)
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)

    heads = None
    param_groups = [{"params": model.parameters(), "lr": LR, "weight_decay": 0.01}]
    if use_heads:
        heads = DisentangleHeads(
            model.config.hidden_size, HEAD_PROJ_DIM,
            use_grl=(STRATEGY == "adversarial"), grl_lam=GRL_LAMBDA
        ).to(DEVICE)
        param_groups.append({"params": heads.parameters(), "lr": HEAD_LR, "weight_decay": 0.01})
        print(f"Head params: {sum(p.numel() for p in heads.parameters()):,}")

    optimizer = torch.optim.AdamW(param_groups)
    total_steps = (len(dataloader) * EPOCHS) // GRAD_ACCUM
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)

    # [FIX 2] Initialize loss normalizer
    loss_keys = ["cls", "con", "vh", "ph", "ort", "dec"]
    loss_norm = LossNormalizer(loss_keys)

    print(f"Dataset: {len(dataset)} examples, {len(dataloader)} batches/epoch")
    print(f"Total steps: {total_steps}\n")

    history = []

    # ---- EVAL BEFORE TRAINING (epoch 0 baseline) ----
    baseline_entry = {"epoch": 0, "cls": None, "total": None, "note": "pre-training baseline"}
    run_checkpoint_eval(model, tokenizer, "epoch0_baseline", baseline_entry)
    history.append(baseline_entry)

    model.train()
    if heads: heads.train()

    for epoch in range(EPOCHS):
        m = {"cls": 0, "con": 0, "val_h": 0, "plaus_h": 0, "ort": 0, "dec": 0, "total": 0, "n": 0}

        for bi, batch in enumerate(dataloader):
            # ---- SIMPLE / ORTHOGONAL / ADVERSARIAL ----
            if not use_pairs:
                ids = batch["input_ids"].to(DEVICE)
                mask = batch["attention_mask"].to(DEVICE)
                plen = batch["prompt_len"]
                val_lbl = batch["validity"].to(DEVICE)
                plaus_lbl = batch["plausibility"].to(DEVICE)

                out = model(ids, attention_mask=mask,
                            output_hidden_states=use_heads,
                            output_attentions=need_attn,
                            use_cache=False)

                cls = causal_lm_loss(out.logits, ids, mask, plen)
                # [FIX 2] Normalize before weighting
                loss = loss_norm.normalize("cls", cls) * 1.0
                m["cls"] += cls.item() * ids.shape[0]

                if use_heads:
                    attns = out.attentions if need_attn else None
                    rep = pool_repr(out.hidden_states, mask, EXTRACT_LAYER, POOL_MODE, attns)
                    vl, pl, hv, hp = heads(rep)
                    vh_loss = F.cross_entropy(vl, val_lbl)
                    ph_loss = F.cross_entropy(pl, plaus_lbl)
                    ol = orth_loss(hv, hp)
                    dl = decorr_loss(hv, hp)

                    # [FIX 2] Each component normalized then weighted
                    loss = loss + LAMBDA_VAL_HEAD * loss_norm.normalize("vh", vh_loss)
                    loss = loss + LAMBDA_PLAUS * loss_norm.normalize("ph", ph_loss)
                    loss = loss + LAMBDA_ORTH * loss_norm.normalize("ort", ol)
                    loss = loss + LAMBDA_DECORR * loss_norm.normalize("dec", dl)

                    m["val_h"] += vh_loss.item() * ids.shape[0]
                    m["plaus_h"] += ph_loss.item() * ids.shape[0]
                    m["ort"] += ol.item() * ids.shape[0]
                    m["dec"] += dl.item() * ids.shape[0]

                m["total"] += loss.item() * ids.shape[0]
                m["n"] += ids.shape[0]

            # ---- CONTRASTIVE ----
            else:
                ids_a = batch["input_ids_a"].to(DEVICE)
                mask_a = batch["attention_mask_a"].to(DEVICE)
                plen_a = batch["prompt_len_a"]
                ids_b = batch["input_ids_b"].to(DEVICE)
                mask_b = batch["attention_mask_b"].to(DEVICE)
                plen_b = batch["prompt_len_b"]
                pair_type = batch["pair_type"].to(DEVICE)

                # Sequential forward passes to halve peak memory.
                out_a = model(ids_a, attention_mask=mask_a, output_hidden_states=True, use_cache=False)
                cls_a = causal_lm_loss(out_a.logits, ids_a, mask_a, plen_a)
                hidden_a = tuple(h.detach() for h in out_a.hidden_states)
                del out_a
                torch.cuda.empty_cache()

                out_b = model(ids_b, attention_mask=mask_b, output_hidden_states=True, use_cache=False)
                cls_b = causal_lm_loss(out_b.logits, ids_b, mask_b, plen_b)
                hidden_b = tuple(h.detach() for h in out_b.hidden_states)
                del out_b
                torch.cuda.empty_cache()

                cls = (cls_a + cls_b) / 2.0

                con = contrastive_repr_loss(
                    hidden_a, hidden_b,
                    mask_a, mask_b, CONTRASTIVE_LAYER_FRAC, pair_type
                )
                del hidden_a, hidden_b

                # [FIX 2] Normalized loss mixing
                loss = loss_norm.normalize("cls", cls) * 1.0 + LAMBDA_CONTRASTIVE * loss_norm.normalize("con", con)

                bs = ids_a.shape[0]
                m["cls"] += cls.item() * bs
                m["con"] += con.item() * bs
                m["total"] += loss.item() * bs
                m["n"] += bs

            (loss / GRAD_ACCUM).backward()

            if (bi + 1) % GRAD_ACCUM == 0:
                params = list(model.parameters()) + (list(heads.parameters()) if heads else [])
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()

            if (bi + 1) % 30 == 0:
                n = m["n"]
                parts = [f"cls={m['cls']/n:.4f}"]
                if use_pairs: parts.append(f"con={m['con']/n:.4f}")
                if use_heads: parts.extend([f"vh={m['val_h']/n:.4f}", f"ph={m['plaus_h']/n:.4f}", f"ort={m['ort']/n:.6f}"])
                # [FIX 2] Show EMA scales so you can monitor normalization
                scales = " | scales: " + " ".join(f"{k}={loss_norm.ema[k]:.4f}" for k in loss_keys if loss_norm.initialized[k])
                print(f"  E{epoch+1} [{bi+1}/{len(dataloader)}] {' '.join(parts)}{scales}")

        n = m["n"]
        ep = {"epoch": epoch + 1, "cls": m["cls"]/n, "total": m["total"]/n}
        if use_pairs: ep["contrastive"] = m["con"]/n
        if use_heads: ep.update({"val_head": m["val_h"]/n, "plaus_head": m["plaus_h"]/n, "orth": m["ort"]/n})
        print(f"\n  Epoch {epoch+1} train losses: {ep}\n")

        ckpt = os.path.join(OUTPUT_DIR, f"epoch{epoch+1}")
        model.save_pretrained(ckpt); tokenizer.save_pretrained(ckpt)
        if heads: torch.save(heads.state_dict(), os.path.join(ckpt, "heads.pt"))

        # ---- EVAL AFTER EACH EPOCH ----
        run_checkpoint_eval(model, tokenizer, f"epoch{epoch+1}", ep)
        if heads: heads.train()

        history.append(ep)

    final = os.path.join(OUTPUT_DIR, "final")
    model.save_pretrained(final); tokenizer.save_pretrained(final)
    if heads: torch.save(heads.state_dict(), os.path.join(final, "heads.pt"))
    save_json(history, os.path.join(OUTPUT_DIR, "history.json"))
    print(f"\nSaved to {final}")

    # Print summary table
    print(f"\n{'='*70}")
    print(f"  TRAINING SUMMARY: ACC / CE / Score across epochs")
    print(f"{'='*70}")
    for h in history:
        e = h.get("epoch", "?")
        acc = h.get("accuracy", "N/A")
        ce = h.get("content_effect", "N/A")
        sc = h.get("combined_score", "N/A")
        acc_s = f"{acc:.4f}" if isinstance(acc, float) else acc
        ce_s = f"{ce:.4f}" if isinstance(ce, float) else ce
        sc_s = f"{sc:.4f}" if isinstance(sc, float) else sc
        print(f"  Epoch {e:>2s}:  ACC={acc_s}  CE={ce_s}  Score={sc_s}" if isinstance(e, str) else
              f"  Epoch {e:>2d}:  ACC={acc_s}  CE={ce_s}  Score={sc_s}")
    print(f"{'='*70}\n")

    return model, tokenizer, heads


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    model, tokenizer, heads = train()

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

trainable params: 5,046,272 || all params: 7,620,662,784 || trainable%: 0.0662
  [GPU after model+lora] allocated=14.20GB reserved=14.22GB
Pairs: 474 positive (push together), 474 negative (push apart)
Dataset: 948 examples, 948 batches/epoch
Total steps: 296


  EVALUATION: epoch0_baseline
  50/191
  100/191
  150/191
Predictions: lora_contrastive_output/predictions_epoch0_baseline.json

Subgroup accuracy:
  implausible_invalid           :  97.92% (47/48)
  implausible_valid             :  54.17% (26/48)
  plausible_invalid             :  55.32% (26/47)
  plausible_valid               :  97.92% (47/48)
Scoring complete. Results written to lora_contrastive_output/eval_results.json
ACC=76.44  TCE=43.1738  Score=15.9644



/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


  E1 [30/948] cls=17.2490 con=0.3046 | scales: cls=17.2490 con=0.3046
  E1 [60/948] cls=17.2188 con=0.3288 | scales: cls=16.0212 con=0.3883
  E1 [90/948] cls=17.1854 con=0.3443 | scales: cls=17.7027 con=0.2592
  E1 [120/948] cls=17.5203 con=0.3331 | scales: cls=17.4066 con=0.2175
  E1 [150/948] cls=17.3096 con=0.3282 | scales: cls=15.9229 con=0.2826
  E1 [180/948] cls=17.0370 con=0.3566 | scales: cls=14.1978 con=0.5108
  E1 [210/948] cls=16.6297 con=0.3563 | scales: cls=13.8041 con=0.3651
  E1 [240/948] cls=16.0986 con=0.3746 | scales: cls=11.8656 con=0.4613
  E1 [270/948] cls=15.4017 con=0.3863 | scales: cls=9.6728 con=0.4675
  E1 [300/948] cls=14.4114 con=0.3807 | scales: cls=4.9682 con=0.3423
  E1 [330/948] cls=13.4340 con=0.3868 | scales: cls=3.2068 con=0.5259
  E1 [360/948] cls=12.4355 con=0.3837 | scales: cls=1.3700 con=0.4079
  E1 [390/948] cls=11.5419 con=0.3828 | scales: cls=0.7922 con=0.3385
  E1 [420/948] cls=10.7441 con=0.3834 | scales: cls=0.5682 con=0.3820
  E1 [450/948] 